# Appendix 3.F — Synthetic Query Formulation Comparison (Facial Skincare)

This notebook compares four synthetic-query formulations under a matched retrieval evaluation.

| Variant | Formulation | Role |
|---|---|---|
| **A** | Deterministic seed assembled from review-safe signals | Review-safe baseline |
| **B** | Free-form LLM rewrite of A | Comparison variant |
| **C** | Constrained DSPy linguistic rewrite | Adopted benchmark query |
| **D** | HyDE-style expansion of C | Comparison variant |

All formulations are evaluated on the same 2,288 cases, item universe, four query-only retrieval methods, and exact 1,000-candidate budget. The notebook reports HitRate across candidate depths, headline HitRate@1,000, query audits, amortized runtime summaries, paired comparisons against formulation C, and retrieval-reproduction checks.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
%pip -q install faiss-cpu rank_bm25 sentence-transformers dspy-ai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 19.3 MB/s eta 0:00:00


In [3]:
# =========================================================
# Config
# =========================================================
# Reuse the production retrieval definitions and add only
# the controls required for this appendix comparison.
from pathlib import Path
from collections import defaultdict
import heapq
import json
import math
import re
import time
import types

import faiss
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 240)

NOTEBOOK_NAME = "06x_query_variant_retrieval_comparison_face.ipynb"
CATEGORY_ID = "face"
CATEGORY_LABEL = "Facial Skincare"
PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/facial_skincare")

# ---- Inputs (all read-only) -----------------------------------------------
QUERY_CACHE_PATH = PROJECT_ROOT / "outputs/query_cache/face_queries.parquet"
QUERY_CONTRACT_PATH = PROJECT_ROOT / "outputs/query_summary/face_queries_config.json"
ITEM_DOCS_PATH = PROJECT_ROOT / "data/processed/items/face_item_docs.parquet"
ITEM_FACETS_PATH = PROJECT_ROOT / "data/processed/items/face_items_facets.parquet"
GRAPH_EDGES_PATH = PROJECT_ROOT / "data/processed/items/face_item_graph_edges.parquet"
ITEM_SCHEMA_PATH = PROJECT_ROOT / "data/processed/items/face_item_schema_full.parquet"
SAMPLED_USERS_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_user_regime_sample.parquet"
FINAL_SAMPLING_POOL_PATH = PROJECT_ROOT / "data/processed/user_sampling/face_final_sampling_pool.parquet"
RETRIEVAL_ARTIFACT_MANIFEST_PATH = PROJECT_ROOT / "data/processed/items/retrieval_artifact_manifest_face.json"
SEALED_PER_QUERY_METRICS_PATH = (
    PROJECT_ROOT / "outputs/stage1_query_retrieval_selection/stage1_per_query_metrics_face.parquet"
)

ITEM_ID_COLUMN = "parent_asin"
DENSE_TEXT_COLUMN = "dense_text"
SPARSE_TEXT_COLUMN = "sparse_text"
BRAND_TEXT_COLUMN = "brand_facet_text"
FACET_ITEM_ID_COLUMN = "parent_asin"
FACET_VALUE_COLUMN = "facet_value_norm"
EVIDENCE_SCOPE = "catalog_metadata_functional_facets_and_historical_review_signals"
BRAND_QUERY_MATCHING_ENABLED = False
BRAND_IN_FUNCTIONAL_GRAPH = False

EXPECTED_ROWS = None
EXPECTED_REGIME_COUNTS = None
REGIME_ORDER = ["cold", "weak", "moderate", "strong"]
EXPECTED_TARGET_SELECTION_MODE = "recent_eligible_review_rank_le5"
MAX_TARGET_RANK_ALLOWED = 5
EXPECTED_QUERY_VARIANT = "medium_heavy_review_only_dspy_linguistic"

ACTIVE_QUERY_COLUMN = "query"
COMPATIBILITY_QUERY_ALIAS = None
KNOWN_QUERY_ALIAS_COLUMNS = {"query_C"}
CLEAN_QUERY_COLUMN = "query_clean"
QUERY_METHOD_LABEL = "query"
QUERY_STRUCTURE_COLUMNS = [
    "query_generic_anchor_rate",
    "query_specific_facet_cue_rate",
    "query_specific_facet_family_count",
]
QUERY_PASSTHROUGH_COLUMNS = ["sampling_bracket"]
# "query_seed" is variant A and must be loaded; it is absent from the sealed
# Notebook 07 audit list because that notebook never reads the seed.
OPTIONAL_QUERY_AUDIT_COLUMNS = [
    CLEAN_QUERY_COLUMN,
    "query_clean_is_active",
    "query_generic_anchor_terms",
    "query_generic_utility_terms",
    "query_generation_status",
    "replacement_case_used",
    "replacement_source_case_id",
    "query_seed",
    "query_dspy_status",
    "query_fallback_used",
]

METHOD_KEYS = [
    "dense_bert",
    "bm25",
    "hybrid_dense_bm25",
    "graph_hybrid",
]
METHOD_DISPLAY_NAMES = {
    "dense_bert": "Dense BERT",
    "bm25": "BM25",
    "hybrid_dense_bm25": "Dense-BM25 Hybrid",
    "graph_hybrid": "Graph-Hybrid",
}

RRF_K = 60
HYBRID_WEIGHTS = {"dense": 1.00, "bm25": 1.00}
GRAPH_HYBRID_WEIGHTS = {
    "dense": 0.90,
    "bm25": 1.00,
    "metadata_graph": 0.50,
    "review_reputation_graph": 0.35,
}

EVAL_KS = [1, 5, 10, 100, 300, 500, 700, 1000]
MAX_RETRIEVAL_K = 1000
HEADLINE_DEPTH = 1000
FIGURE_DEPTHS = [100, 300, 500, 700, 1000]
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
ITEM_EMBEDDING_BATCH_SIZE = 128
QUERY_EMBEDDING_BATCH_SIZE = 128

RUN_SMOKE_TEST = False
SMOKE_TEST_N = 20
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

TOKEN_EQUIVALENCE = {
    "sensitive": "sensitivity", "sensitivity": "sensitivity",
    "oily": "oiliness", "oiliness": "oiliness",
    "dry": "dryness", "dryness": "dryness",
    "wrinkle": "wrinkles", "wrinkles": "wrinkles",
    "hydrating": "hydration", "hydrate": "hydration",
    "hydration": "hydration", "hydrated": "hydration",
    "moisturizer": "moisturizer", "moisturizers": "moisturizer",
    "moisturizing": "moisturizer", "moisturize": "moisturizer",
    "cream": "moisturizer", "creams": "moisturizer",
    "lotion": "moisturizer", "lotions": "moisturizer",
    "serum": "serum", "serums": "serum",
    "mask": "mask", "masks": "mask",
    "cleanser": "cleanser", "cleansers": "cleanser",
    "cleansing": "cleanser", "wash": "cleanser",
    "toner": "toner", "toners": "toner",
    "treatment": "treatment", "treatments": "treatment",
    "acne": "acne", "blemish": "acne", "blemishes": "acne",
    "breakout": "acne", "breakouts": "acne",
    "pore": "pores", "pores": "pores",
    "brightening": "brightening", "brighten": "brightening",
    "brightness": "brightening",
    "hyperpigmentation": "dark_spots", "pigmentation": "dark_spots",
    "spots": "dark_spots", "spot": "dark_spots", "dark": "dark",
}
LINGUISTIC_STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from",
    "in", "into", "is", "it", "of", "on", "or", "that", "the", "this",
    "to", "with", "without", "you", "your",
}
GENERIC_ANCHOR_TOKENS = {
    "product", "products", "item", "items", "solution", "solutions",
    "face", "facial", "skin", "skincare", "care", "routine", "support",
}
GENERIC_UTILITY_TOKENS = {
    "support", "wellness", "natural", "formula", "blend", "complex",
    "routine", "care", "product", "products", "solution", "solutions",
}
GRAPH_STOPWORDS = (
    LINGUISTIC_STOPWORDS
    | GENERIC_ANCHOR_TOKENS
    | GENERIC_UTILITY_TOKENS
    | {"query", "using", "use"}
)
GRAPH_SHORT_TOKENS = {"b5", "e", "c"}
MIN_GRAPH_TOKEN_LENGTH = 3
GRAPH_MAX_PHRASE_TOKENS = 5
GRAPH_PHRASE_BONUS = 1.25
FORBIDDEN_FACET_SOURCE_TERMS = ("identifier", "count", "policy", "diagnostic")

# ---- Variant axis ----------------------------------------------------------
VARIANT_KEYS = ["A", "B", "C", "D"]
VARIANT_LABELS = {
    "A": "A. Rule-based baseline",
    "B": "B. LLM rewrite",
    "C": "C. DSPy retrieval query (adopted)",
    "D": "D. DSPy + HyDE-style query",
}
VARIANT_SOURCE = {
    "A": "sealed_cache_query_seed",
    "B": "generated_llm_rewrite_of_seed",
    "C": "sealed_cache_active_query",
    "D": "generated_hyde_expansion_of_C",
}
VARIANT_LEXICAL_FREEDOM = {"A": "none", "B": "free_short", "C": "seed_concept_set", "D": "free_paragraph"}
ADOPTED_VARIANT = "C"
GENERATED_VARIANTS = ["B", "D"]

B_MIN_TOKENS, B_MAX_TOKENS = 5, 30
D_MIN_TOKENS, D_MAX_TOKENS = 20, 80   # pre-declared cap; bounds the BM25 posting-scan cost

# ---- Generation (identical endpoint and secret to Notebook 06) -------------
USE_LLM_GENERATION = True
FAIL_IF_LLM_UNAVAILABLE = True
DEEPSEEK_MODEL = "openai/deepseek-chat"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
DEEPSEEK_COLAB_SECRET = "deepseek_api_key"
DSPY_TEMPERATURE = 0.0
DSPY_MAX_TOKENS = {"B": 96, "D": 220}
DSPY_MAX_RETRIES = 2
DSPY_RETRY_SLEEP_SECONDS = 1.5

# ---- Pre-declared inference settings (fixed before any result is observed) --
NON_INFERIORITY_MARGIN_PP = 1.0     # variant A vs adopted variant C, HitRate@1000
BOOTSTRAP_REPLICATES = 10000
BOOTSTRAP_SEED = 42
REPRODUCTION_GATE_MODE = "hit_indicator"
REPRODUCTION_GATE_BLOCKING = False   # variant C is not required to reproduce byte-identically

# ---- Outputs (disposable; nothing else may be written) ---------------------
OUT_DIR = PROJECT_ROOT / "outputs/analysis/_disposable_query_variant_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GENERATION_CACHE_PATH = OUT_DIR / "variant_generated_queries_face.parquet"
VARIANT_TEXT_AUDIT_PATH = OUT_DIR / "variant_text_audit_face.csv"
LEAKAGE_AUDIT_PATH = OUT_DIR / "variant_leakage_flag_audit_face.csv"
PER_QUERY_METRICS_PATH = OUT_DIR / "variant_per_query_metrics_face.parquet"
POOL_DEPTH_SUMMARY_PATH = OUT_DIR / "variant_method_pool_depth_summary.csv"
HEADLINE_SUMMARY_PATH = OUT_DIR / "variant_method_headline_hitrate.csv"
RUNTIME_COMPONENTS_PATH = OUT_DIR / "variant_runtime_components.csv"
RUNTIME_METHOD_PATH = OUT_DIR / "variant_method_runtime.csv"
PAIRED_INFERENCE_PATH = OUT_DIR / "variant_paired_inference_vs_C.csv"
REPRODUCTION_REPORT_PATH = OUT_DIR / "variant_C_reproduction_report.csv"
RUN_MANIFEST_PATH = OUT_DIR / "variant_comparison_run_manifest.json"

EMITTED_PATHS = [
    GENERATION_CACHE_PATH, VARIANT_TEXT_AUDIT_PATH, LEAKAGE_AUDIT_PATH,
    PER_QUERY_METRICS_PATH, POOL_DEPTH_SUMMARY_PATH, HEADLINE_SUMMARY_PATH,
    RUNTIME_COMPONENTS_PATH, RUNTIME_METHOD_PATH, PAIRED_INFERENCE_PATH,
    REPRODUCTION_REPORT_PATH, RUN_MANIFEST_PATH,
]
for _path in EMITTED_PATHS:
    if OUT_DIR not in _path.parents:
        raise RuntimeError(f"Emission escapes the disposable directory: {_path}")

print("Input :", QUERY_CACHE_PATH)
print("Input :", ITEM_DOCS_PATH)
print("Input :", ITEM_SCHEMA_PATH)
print("Output:", OUT_DIR)
print("Variants:", ", ".join(VARIANT_LABELS[key] for key in VARIANT_KEYS))
print("Headline depth:", HEADLINE_DEPTH, "| non-inferiority margin (pp):", NON_INFERIORITY_MARGIN_PP)


Input : /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/query_cache/face_queries.parquet
Input : /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_item_docs.parquet
Input : /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/data/processed/items/face_item_schema_full.parquet
Output: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/_disposable_query_variant_comparison
Variants: A. Rule-based baseline, B. LLM rewrite, C. DSPy retrieval query (adopted), D. DSPy + HyDE-style query
Headline depth: 1000 | non-inferiority margin (pp): 1.0


In [4]:
# Reuse the production retrieval helper body unchanged.
# =========================================================
# Shared Functions
# =========================================================
def normalize_space(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\n", " ").replace("\t", " ")).strip()


def canonical_tokens(value):
    text = normalize_space(value).lower().replace("_", " ").replace("-", " ").replace("/", " ")
    tokens = re.findall(r"[a-z0-9']+", text)
    return [TOKEN_EQUIVALENCE.get(token, token) for token in tokens]


def canonical_text(value):
    return " ".join(canonical_tokens(value))


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def require_columns(frame, required, frame_name):
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise RuntimeError(f"{frame_name} is missing required columns: {missing}")


def boolean_series(values):
    if pd.api.types.is_bool_dtype(values):
        return values.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(values):
        return values.fillna(0).astype(float).ne(0)
    normalized = values.fillna("").astype(str).str.strip().str.lower()
    return normalized.isin({"1", "true", "t", "yes", "y"})


def validate_query_contract(contract):
    if not isinstance(contract, dict) or not contract:
        raise RuntimeError("Notebook 06 query contract must be a non-empty JSON object.")
    if contract.get("active_query_column") != ACTIVE_QUERY_COLUMN:
        raise RuntimeError("Notebook 06 active_query_column must be query.")
    if contract.get("evidence_scope") != "target_review_safe_signals_only":
        raise RuntimeError("Notebook 06 evidence_scope must be target_review_safe_signals_only.")
    if contract.get("query_evidence_source") != "target_review_safe_signals_only":
        raise RuntimeError("Notebook 06 query_evidence_source mismatch.")
    if contract.get("query_variant") != EXPECTED_QUERY_VARIANT:
        raise RuntimeError("Notebook 06 query_variant mismatch.")
    contract_target_per_regime = int(contract.get("target_per_regime", -1))
    if contract_target_per_regime <= 0:
        raise RuntimeError("Notebook 06 target_per_regime must be positive.")
    if list(contract.get("regime_order", [])) != REGIME_ORDER:
        raise RuntimeError("Notebook 06 regime_order mismatch.")
    expected_false = [
        "target_metadata_fallback_used",
        "item_context_fallback_used",
        "item_metadata_evidence_used",
        "historical_review_evidence_used",
        "user_prior_evidence_used",
        "rating_evidence_used",
        "sentiment_evidence_used",
        "query_clean_is_active",
    ]
    bad_flags = [key for key in expected_false if contract.get(key) is not False]
    if bad_flags:
        raise RuntimeError(f"Notebook 06 contract flags must be false: {bad_flags}")
    if contract.get("user_prior_columns_loaded") not in (None, []):
        raise RuntimeError("Notebook 06 must not load user-prior columns.")
    if contract.get("user_prior_columns_exported") not in (None, []):
        raise RuntimeError("Notebook 06 must not export user-prior columns.")
    aliases = contract.get("compatibility_query_aliases", {})
    if COMPATIBILITY_QUERY_ALIAS is None:
        if aliases not in ({}, None):
            raise RuntimeError("Unexpected Notebook 06 compatibility query alias.")
    elif aliases != {COMPATIBILITY_QUERY_ALIAS: ACTIVE_QUERY_COLUMN}:
        raise RuntimeError("Notebook 06 compatibility query alias mismatch.")


def validate_retrieval_manifest(manifest):
    if not isinstance(manifest, dict) or not manifest:
        raise RuntimeError("Notebook 04 retrieval manifest must be a non-empty JSON object.")
    expected = {
        "evidence_scope": EVIDENCE_SCOPE,
        "historical_review_reputation_enabled": True,
        "review_reputation_graph_enabled": True,
        "brand_graph_enabled": True,
        "brand_in_retrieval_text": True,
        "brand_in_profile_source_text": True,
        "brand_in_synthetic_query": False,
        "raw_review_text_exported": False,
        "dense_source": DENSE_TEXT_COLUMN,
        "sparse_source": SPARSE_TEXT_COLUMN,
        "item_facet_export_option": "B_all_rows_with_reliable_flags",
    }
    mismatches = {
        key: manifest.get(key)
        for key, expected_value in expected.items()
        if manifest.get(key) != expected_value
    }
    if mismatches:
        raise RuntimeError(f"Notebook 04 retrieval manifest mismatch: {mismatches}")
    brand_in_functional_graph = manifest.get(
        "brand_in_functional_graph",
        BRAND_IN_FUNCTIONAL_GRAPH,
    )
    if brand_in_functional_graph is not BRAND_IN_FUNCTIONAL_GRAPH:
        raise RuntimeError(
            "Notebook 04 manifest must declare or imply brand_in_functional_graph=False."
        )
    if not normalize_space(manifest.get("brand_graph_mask")):
        raise RuntimeError("Notebook 04 manifest must document the brand graph mask.")
    if not normalize_space(manifest.get("global_review_graph_mask")):
        raise RuntimeError("Notebook 04 manifest must document the Global Review graph mask.")


def validate_false_columns(frame, columns, frame_name):
    for column in columns:
        if boolean_series(frame[column]).any():
            raise RuntimeError(f"{frame_name}.{column} must be false for every row.")


GRAPH_FACET_FLAG_COLUMNS = [
    "is_product_functional_facet",
    "is_query_safe",
    "is_brand",
    "is_review_derived",
    "is_generic_category_anchor",
    "is_generic_utility_token",
    "is_context_dependent_utility_token",
    "is_metadata_facet_source",
    "is_disallowed_nonfacet_source",
    "is_core_graph_facet",
    "is_brand_graph_facet",
    "is_retrieval_safe",
    "is_profile_safe",
    "is_global_review_graph_facet",
]


def normalize_facet_flags(frame):
    normalized = frame.copy()
    for column in GRAPH_FACET_FLAG_COLUMNS:
        normalized[column] = boolean_series(normalized[column])
    return normalized


def explicit_metadata_facet_mask(facets):
    return (
        facets["is_product_functional_facet"]
        & facets["is_query_safe"]
        & ~facets["is_brand"]
        & ~facets["is_review_derived"]
        & ~facets["is_generic_category_anchor"]
        & ~facets["is_generic_utility_token"]
        & ~facets["is_context_dependent_utility_token"]
        & facets["is_metadata_facet_source"]
        & ~facets["is_disallowed_nonfacet_source"]
    )


def explicit_review_facet_mask(facets):
    return (
        facets["is_review_derived"]
        & ~facets["is_brand"]
        & ~facets["is_generic_category_anchor"]
        & ~facets["is_generic_utility_token"]
        & ~facets["is_context_dependent_utility_token"]
        & ~facets["is_disallowed_nonfacet_source"]
    )


def explicit_brand_facet_mask(facets):
    return (
        facets["facet_role"].eq("brand")
        & facets["is_brand"]
        & ~facets["is_review_derived"]
        & ~facets["is_generic_category_anchor"]
        & ~facets["is_generic_utility_token"]
        & ~facets["is_context_dependent_utility_token"]
        & facets["is_metadata_facet_source"]
        & ~facets["is_disallowed_nonfacet_source"]
    )

def explicit_global_review_facet_mask(facets):
    return (
        explicit_metadata_facet_mask(facets)
        | explicit_brand_facet_mask(facets)
        | explicit_review_facet_mask(facets)
    )


def audit_global_review_facet_flags(raw_facet_flags):
    required = [FACET_ITEM_ID_COLUMN, "source_column", "facet_role", *GRAPH_FACET_FLAG_COLUMNS]
    require_columns(raw_facet_flags, required, "item facet flags")
    facets = normalize_facet_flags(raw_facet_flags[required])

    metadata_mask = explicit_metadata_facet_mask(facets)
    brand_mask = explicit_brand_facet_mask(facets)
    review_mask = explicit_review_facet_mask(facets)
    global_mask = metadata_mask | brand_mask | review_mask

    core_mismatch_rows = int(facets["is_core_graph_facet"].ne(metadata_mask).sum())
    brand_mismatch_rows = int(facets["is_brand_graph_facet"].ne(brand_mask).sum())
    global_mismatch_rows = int(facets["is_global_review_graph_facet"].ne(global_mask).sum())
    if core_mismatch_rows:
        raise RuntimeError(
            f"is_core_graph_facet disagrees with the explicit metadata mask: {core_mismatch_rows}"
        )
    if brand_mismatch_rows:
        raise RuntimeError(
            f"is_brand_graph_facet disagrees with the explicit brand mask: {brand_mismatch_rows}"
        )
    if global_mismatch_rows:
        raise RuntimeError(
            f"is_global_review_graph_facet disagrees with the explicit Global Review mask: {global_mismatch_rows}"
        )

    metadata_rows = facets.loc[metadata_mask].copy()
    brand_rows = facets.loc[brand_mask].copy()
    review_rows = facets.loc[review_mask].copy()
    global_rows = facets.loc[global_mask].copy()
    if metadata_rows.empty:
        raise RuntimeError("Metadata product-functional facet rows must be greater than zero.")
    if brand_rows.empty:
        raise RuntimeError("Brand facet rows must be greater than zero.")
    if review_rows.empty:
        raise RuntimeError("Historical review-derived facet rows must be greater than zero.")
    if global_rows.empty:
        raise RuntimeError("Global Review graph facet rows must be greater than zero.")

    source_text = metadata_rows["source_column"].fillna("").astype(str).str.lower()
    disallowed_source_rows = source_text.map(
        lambda value: any(term in value for term in FORBIDDEN_FACET_SOURCE_TERMS)
    )
    if disallowed_source_rows.any():
        raise RuntimeError(
            "Identifier, count, policy, or diagnostic sources entered the metadata graph mask."
        )
    if brand_rows["is_query_safe"].any():
        raise RuntimeError("Brand rows must remain query-unsafe.")
    if brand_rows["is_product_functional_facet"].any():
        raise RuntimeError("Brand rows must remain separate from product-functional facets.")
    if not brand_rows["is_retrieval_safe"].all():
        raise RuntimeError("Brand rows must be retrieval-safe.")
    if not brand_rows["is_profile_safe"].all():
        raise RuntimeError("Brand rows must be profile-safe.")
    if review_rows["is_product_functional_facet"].any():
        raise RuntimeError("Historical review-derived rows must not be product-functional facets.")
    if review_rows["is_query_safe"].any():
        raise RuntimeError("Historical review-derived rows must not be query-safe facets.")

    return pd.DataFrame([{
        "raw_facet_rows": int(len(facets)),
        "global_review_flagged_rows": int(len(global_rows)),
        "metadata_functional_rows": int(len(metadata_rows)),
        "brand_rows_used": int(len(brand_rows)),
        "review_derived_rows_used": int(len(review_rows)),
        "generic_anchor_rows_used": int(global_rows["is_generic_category_anchor"].sum()),
        "generic_utility_rows_used": int(global_rows["is_generic_utility_token"].sum()),
        "context_utility_rows_used": int(global_rows["is_context_dependent_utility_token"].sum()),
        "disallowed_source_rows_used": int(global_rows["is_disallowed_nonfacet_source"].sum()),
        "core_mask_mismatch_rows": core_mismatch_rows,
        "brand_mask_mismatch_rows": brand_mismatch_rows,
        "global_mask_mismatch_rows": global_mismatch_rows,
        "historical_facet_values_loaded": int(len(review_rows)),
        "brand_facet_values_loaded": int(len(brand_rows)),
        "brand_query_matching_enabled": False,
        "global_value_load_filter": "is_global_review_graph_facet == True",
        "item_facet_export_option": "B_all_rows_with_reliable_flags",
        "status": "PASS",
    }])

def canonicalize_graph_facets(raw_facets, item_to_position, facet_kind):
    required = [
        FACET_ITEM_ID_COLUMN,
        FACET_VALUE_COLUMN,
        "source_column",
        "facet_role",
        *GRAPH_FACET_FLAG_COLUMNS,
    ]
    require_columns(raw_facets, required, f"{facet_kind} item facets")
    facets = normalize_facet_flags(raw_facets[required])
    if facets.empty:
        raise RuntimeError(f"{facet_kind} item-facet value rows must be greater than zero.")

    masks = {
        "metadata": explicit_metadata_facet_mask,
        "brand": explicit_brand_facet_mask,
        "review": explicit_review_facet_mask,
    }
    if facet_kind not in masks:
        raise RuntimeError(f"Unsupported graph facet kind: {facet_kind}")
    expected_mask = masks[facet_kind](facets)
    if not expected_mask.all():
        raise RuntimeError(f"A loaded {facet_kind} facet row violates its explicit mask.")
    if not facets["is_global_review_graph_facet"].all():
        raise RuntimeError(f"A loaded {facet_kind} facet row is outside the Global Review graph mask.")

    facets["item_id"] = facets[FACET_ITEM_ID_COLUMN].astype(str)
    facets["item_position"] = facets["item_id"].map(item_to_position)
    if facets["item_position"].isna().any():
        raise RuntimeError(f"{facet_kind} facets contain item IDs missing from item docs.")
    facets["item_position"] = facets["item_position"].astype(np.int32)
    facets["facet_value_norm"] = facets[FACET_VALUE_COLUMN].map(canonical_text)
    facets = facets.loc[facets["facet_value_norm"].str.len().gt(0)].copy()
    facets = facets.drop_duplicates(["item_position", "facet_value_norm"]).reset_index(drop=True)
    if facets.empty:
        raise RuntimeError(f"No non-empty {facet_kind} facet values remain after normalization.")
    return facets


def validate_global_review_graph_edges(graph_edges):
    required = [
        "source_node_type",
        "target_node_type",
        "source_column",
        "facet_role",
        *GRAPH_FACET_FLAG_COLUMNS,
    ]
    require_columns(graph_edges, required, "Global Review graph edges")
    graph_edges = normalize_facet_flags(graph_edges[required])

    metadata_mask = explicit_metadata_facet_mask(graph_edges)
    brand_mask = explicit_brand_facet_mask(graph_edges)
    review_mask = explicit_review_facet_mask(graph_edges)
    global_mask = metadata_mask | brand_mask | review_mask

    if not graph_edges["is_core_graph_facet"].eq(metadata_mask).all():
        raise RuntimeError("Graph-edge is_core_graph_facet disagrees with the metadata mask.")
    if not graph_edges["is_brand_graph_facet"].eq(brand_mask).all():
        raise RuntimeError("Graph-edge is_brand_graph_facet disagrees with the brand mask.")
    if not graph_edges["is_global_review_graph_facet"].eq(global_mask).all():
        raise RuntimeError("Graph-edge is_global_review_graph_facet disagrees with the Global Review mask.")
    if not graph_edges["is_global_review_graph_facet"].all():
        raise RuntimeError("The production graph contains a row outside the Global Review graph mask.")
    if not graph_edges["source_node_type"].eq("item").all():
        raise RuntimeError("Production graph source nodes must be items.")
    if not graph_edges["target_node_type"].eq("entity").all():
        raise RuntimeError("Production graph target nodes must be entities.")
    if int(metadata_mask.sum()) <= 0:
        raise RuntimeError("Metadata graph edges must be greater than zero.")
    if int(brand_mask.sum()) <= 0:
        raise RuntimeError("Brand graph edges must be greater than zero.")
    if int(review_mask.sum()) <= 0:
        raise RuntimeError("Historical review graph edges must be greater than zero.")
    if int(global_mask.sum()) != len(graph_edges):
        raise RuntimeError("Global Review graph edge decomposition is incomplete.")

    for column in [
        "is_generic_category_anchor",
        "is_generic_utility_token",
        "is_context_dependent_utility_token",
        "is_disallowed_nonfacet_source",
    ]:
        if graph_edges[column].any():
            raise RuntimeError(f"Production graph contains prohibited rows: {column}")
    if not graph_edges.loc[metadata_mask, "is_product_functional_facet"].all():
        raise RuntimeError("Metadata graph rows must be product-functional.")
    if not graph_edges.loc[metadata_mask, "is_query_safe"].all():
        raise RuntimeError("Metadata graph rows must be query-safe.")
    if graph_edges.loc[brand_mask, "is_query_safe"].any():
        raise RuntimeError("Brand graph rows must remain query-unsafe.")
    if graph_edges.loc[brand_mask, "is_product_functional_facet"].any():
        raise RuntimeError("Brand graph rows must remain separate from product-functional facets.")
    if not graph_edges.loc[brand_mask, "is_retrieval_safe"].all():
        raise RuntimeError("Brand graph rows must be retrieval-safe.")
    if not graph_edges.loc[brand_mask, "is_profile_safe"].all():
        raise RuntimeError("Brand graph rows must be profile-safe.")
    if graph_edges.loc[review_mask, "is_product_functional_facet"].any():
        raise RuntimeError("Historical review graph rows must not be product-functional.")
    if graph_edges.loc[review_mask, "is_query_safe"].any():
        raise RuntimeError("Historical review graph rows must not be query-safe.")


def tokenize_sparse_document(value):
    return [token for token in canonical_tokens(value) if token not in LINGUISTIC_STOPWORDS]


def tokenize_sparse_query(value):
    stopwords = LINGUISTIC_STOPWORDS | GENERIC_ANCHOR_TOKENS
    return [token for token in canonical_tokens(value) if token not in stopwords]


def select_graph_query_text(active_query_text):
    tokens = [token for token in canonical_tokens(active_query_text) if token not in GRAPH_STOPWORDS]
    return " ".join(tokens)


def build_graph_index(facets, n_items):
    phrase_items = defaultdict(set)
    token_items = defaultdict(set)
    for row in facets[["item_position", "facet_value_norm"]].itertuples(index=False):
        item_position = int(row.item_position)
        tokens = canonical_tokens(row.facet_value_norm)
        if not tokens:
            continue
        if 2 <= len(tokens) <= GRAPH_MAX_PHRASE_TOKENS:
            phrase_items[" ".join(tokens)].add(item_position)
        for token in dict.fromkeys(tokens):
            if token in GRAPH_STOPWORDS:
                continue
            if len(token) < MIN_GRAPH_TOKEN_LENGTH and token not in GRAPH_SHORT_TOKENS:
                continue
            token_items[token].add(item_position)
    phrase_idf = {
        phrase: math.log((n_items + 1) / (len(items) + 1)) + 1.0
        for phrase, items in phrase_items.items()
    }
    token_idf = {
        token: math.log((n_items + 1) / (len(items) + 1)) + 1.0
        for token, items in token_items.items()
    }
    return {
        "phrase_items": phrase_items,
        "token_items": token_items,
        "phrase_idf": phrase_idf,
        "token_idf": token_idf,
    }


def graph_score_map(raw_query_text, graph_query_text, graph_index):
    raw_tokens = canonical_tokens(raw_query_text)
    graph_tokens = canonical_tokens(graph_query_text)
    matched_raw_positions = set()
    matched_phrase_tokens = set()
    scores = defaultdict(float)

    max_phrase_length = min(GRAPH_MAX_PHRASE_TOKENS, len(raw_tokens))
    for phrase_length in range(max_phrase_length, 1, -1):
        for start in range(len(raw_tokens) - phrase_length + 1):
            positions = set(range(start, start + phrase_length))
            if positions.intersection(matched_raw_positions):
                continue
            phrase_tokens = raw_tokens[start:start + phrase_length]
            if all(token in GRAPH_STOPWORDS for token in phrase_tokens):
                continue
            phrase = " ".join(phrase_tokens)
            postings = graph_index["phrase_items"].get(phrase)
            if not postings:
                continue
            contribution = GRAPH_PHRASE_BONUS * graph_index["phrase_idf"][phrase]
            for item_position in postings:
                scores[item_position] += contribution
            matched_raw_positions.update(positions)
            matched_phrase_tokens.update(phrase_tokens)

    for token in dict.fromkeys(graph_tokens):
        if token in matched_phrase_tokens or token in GRAPH_STOPWORDS:
            continue
        if len(token) < MIN_GRAPH_TOKEN_LENGTH and token not in GRAPH_SHORT_TOKENS:
            continue
        postings = graph_index["token_items"].get(token)
        if not postings:
            continue
        contribution = graph_index["token_idf"][token]
        for item_position in postings:
            scores[item_position] += contribution
    return scores


def top_items_from_scores(score_map, top_k):
    if not score_map:
        return []
    return [
        int(item_position)
        for item_position, _ in heapq.nsmallest(
            min(top_k, len(score_map)),
            score_map.items(),
            key=lambda pair: (-pair[1], pair[0]),
        )
    ]


def bm25_top_exact(bm25, query_tokens, top_k):
    scores = np.asarray(bm25.get_scores(query_tokens), dtype=np.float64)
    if len(scores) == 0:
        return np.array([], dtype=np.int64), np.array([], dtype=np.float64)
    if not np.isfinite(scores).all():
        raise RuntimeError("BM25 scores contain non-finite values.")
    k_eff = min(int(top_k), len(scores))
    stable_position = np.arange(len(scores), dtype=np.int64)
    order = np.lexsort((stable_position, -scores))
    selected = order[:k_eff].astype(np.int64)
    return selected, scores[selected]


class _SyntheticBM25:
    def __init__(self, scores):
        self._scores = np.asarray(scores, dtype=np.float64)

    def get_scores(self, query_tokens):
        return self._scores.copy()


def validate_bm25_top_exact_helper():
    cases = [
        ("zero_positive", [0.0, 0.0, 0.0, 0.0], 3),
        ("fewer_than_k_positive", [3.0, 1.0, 0.0, 0.0], 3),
        ("exactly_k_positive", [3.0, 2.0, 1.0, 0.0], 3),
        ("more_than_k_positive", [4.0, 3.0, 2.0, 1.0], 3),
        ("tied_zero_scores", [0.0, 0.0, 0.0, 0.0, 0.0], 4),
        ("negative_scores", [1.0, 0.0, -0.5, -0.5, -1.0], 5),
        ("catalog_smaller_than_k", [2.0, 0.0], 5),
    ]
    for name, scores, top_k in cases:
        bm25 = _SyntheticBM25(scores)
        first_idx, first_scores = bm25_top_exact(bm25, ["query"], top_k)
        second_idx, second_scores = bm25_top_exact(bm25, ["query"], top_k)
        expected_k = min(top_k, len(scores))
        if len(first_idx) != expected_k:
            raise RuntimeError(f"BM25 exact-K helper failed candidate count test: {name}")
        if not np.array_equal(first_idx, second_idx) or not np.array_equal(first_scores, second_scores):
            raise RuntimeError(f"BM25 exact-K helper is not deterministic: {name}")
        expected_order = np.lexsort((np.arange(len(scores), dtype=np.int64), -np.asarray(scores, dtype=np.float64)))[:expected_k]
        if not np.array_equal(first_idx, expected_order):
            raise RuntimeError(f"BM25 exact-K helper ordering mismatch: {name}")


validate_bm25_top_exact_helper()


def rank_map(items):
    return {int(item): rank for rank, item in enumerate(items, start=1)}


def reciprocal_rank_fusion(rank_maps, weights, top_k):
    scores = defaultdict(float)
    for source_name, source_rank_map in rank_maps.items():
        weight = float(weights[source_name])
        for item_position, rank in source_rank_map.items():
            scores[item_position] += weight / (RRF_K + rank)
    ranked_pairs = heapq.nsmallest(
        min(top_k, len(scores)),
        scores.items(),
        key=lambda pair: (-pair[1], pair[0]),
    )
    return [int(item) for item, _ in ranked_pairs], dict(scores)


def target_rank(ranked_items, target_position):
    for rank, item_position in enumerate(ranked_items, start=1):
        if item_position == target_position:
            return rank
    return None


def metric_record(rank):
    record = {}
    for k in EVAL_KS:
        hit = float(rank is not None and rank <= k)
        record[f"HitRate@{k}"] = hit
        record[f"NDCG@{k}"] = 1.0 / math.log2(rank + 1) if hit else 0.0
        record[f"MRR@{k}"] = 1.0 / rank if hit else 0.0
    return record


def aggregate_metrics(frame, group_columns):
    metric_columns = [
        column for column in frame.columns
        if column.startswith("HitRate@") or column.startswith("NDCG@") or column.startswith("MRR@")
    ]
    grouped = frame.groupby(group_columns, observed=True, dropna=False)
    result = grouped[metric_columns].mean().reset_index()
    counts = grouped.size().rename("n_queries").reset_index()
    return counts.merge(result, on=group_columns, how="left")


def pool_depth_summary(frame, group_columns):
    rows = []
    for group_values, group_frame in frame.groupby(group_columns, observed=True, dropna=False):
        if not isinstance(group_values, tuple):
            group_values = (group_values,)
        group_dict = dict(zip(group_columns, group_values))
        for k in EVAL_KS:
            rows.append({
                **group_dict,
                "candidate_pool_depth": k,
                "n_queries": int(len(group_frame)),
                "HitRate": float(group_frame[f"HitRate@{k}"].mean()),
                "NDCG": float(group_frame[f"NDCG@{k}"].mean()),
                "MRR": float(group_frame[f"MRR@{k}"].mean()),
            })
    return pd.DataFrame(rows)


CANDIDATE_COLUMNS = [
    "case_id",
    "user_id",
    "regime",
    "query_method",
    "method_key",
    "retrieval_method",
    "evidence_scope",
    "target_parent_asin",
    "candidate_parent_asin",
    "candidate_brand_facet_text",
    "candidate_rank",
    "candidate_score",
    "fused_score",
    "score_semantics",
    "dense_rank",
    "bm25_rank",
    "metadata_graph_rank",
    "review_reputation_graph_rank",
    "is_target",
]


def candidate_frame(rows):
    frame = pd.DataFrame(rows, columns=CANDIDATE_COLUMNS)
    string_columns = [
        "case_id", "user_id", "regime", "query_method", "method_key",
        "retrieval_method", "evidence_scope", "target_parent_asin",
        "candidate_parent_asin", "candidate_brand_facet_text", "score_semantics",
    ]
    for column in string_columns:
        frame[column] = frame[column].astype("string")
    for column in [
        "candidate_rank",
        "dense_rank",
        "bm25_rank",
        "metadata_graph_rank",
        "review_reputation_graph_rank",
    ]:
        frame[column] = frame[column].astype("Int32")
    for column in ["candidate_score", "fused_score"]:
        frame[column] = frame[column].astype(np.float32)
    frame["is_target"] = frame["is_target"].astype(bool)
    return frame


In [5]:
# =========================================================
# Target-Metadata Scrub and Leakage Audit (namespaced)
# =========================================================
# Semantics-preserving port of the Notebook 06 direct-cue scrub and leakage audit.
# It is built inside a factory and exposed as SCRUB.* because Notebook 06 and
# Notebook 07 both define a function named `normalize_space` with different
# behaviour; binding both at module scope would let whichever cell ran last
# silently rebind the other. The transplanted Notebook 07 helpers must win at
# module scope, so the Notebook 06 lineage is confined to this namespace.

def _build_scrub_namespace():
    import html as _html
    import re as _re

    METADATA_GENERIC_TOKENS = {
        "skin", "skincare", "care", "face", "facial", "serum", "cream", "gel",
        "lotion", "cleanser", "moisturizer", "mask", "masks", "treatment",
        "treatments", "toner", "essence", "beauty", "cosmetic", "cosmetics",
        "derm", "derma", "dermatology", "labs", "laboratory", "laboratories",
        "official", "store", "shop",
    }
    METADATA_NAME_COLUMNS = ["itemctx_facet_brand_text"]
    METADATA_DIAGNOSTIC_COLUMNS = ["itemctx_identifier_diagnostic_text", "target_parent_asin"]

    ASIN_PATTERN = _re.compile(r"\bB0[A-Z0-9]{8}\b|\bB[0-9A-Z]{9}\b", _re.IGNORECASE)
    PACKAGE_PATTERN = _re.compile(
        r"\b(\d+(?:\.\d+)?\s?(?:oz|fl\.?\s?oz|ml|g|gram|grams|ct|count|pack|packs|pcs|piece|pieces|%|percent|mg|mcg|iu)|spf\s?\d+|asin|seller|manufacturer|barcode|upc)\b",
        _re.IGNORECASE,
    )
    SELLER_PATTERN = _re.compile(
        r"\b(sold by|seller|manufacturer|made by|distributed by|shipped by)\b",
        _re.IGNORECASE,
    )

    def normalize_space(value):
        if value is None or pd.isna(value):
            return ""
        text = _html.unescape(str(value))
        text = _re.sub(r"<br\s*/?>", " ", text, flags=_re.IGNORECASE)
        text = _re.sub(r"<[^>]+>", " ", text)
        return _re.sub(r"\s+", " ", text.replace("\n", " ").replace("\t", " ")).strip()

    def normalize_query_text(value):
        text = normalize_space(value).lower().replace("&", " and ")
        text = _re.sub(r"[^a-z0-9\s\-']", " ", text)
        return _re.sub(r"\s+", " ", text).strip(" -")

    def tokenize(value):
        return _re.findall(r"[a-z0-9']+", normalize_space(value).lower())

    def token_count(value):
        return len(tokenize(value))

    def remove_exact_phrase(text, phrase):
        phrase_tokens = tokenize(phrase)
        if not phrase_tokens:
            return normalize_query_text(text)
        pattern = r"(?<![a-z0-9])" + r"[\s_\-–—]+".join(
            _re.escape(token) for token in phrase_tokens
        ) + r"(?:['’]s)?(?![a-z0-9])"
        return normalize_query_text(
            _re.sub(pattern, " ", normalize_query_text(text), flags=_re.IGNORECASE)
        )

    def exact_phrase_present(text, phrase):
        normalized = normalize_query_text(text)
        return bool(normalized) and remove_exact_phrase(normalized, phrase) != normalized

    def split_metadata_values(value):
        text = normalize_space(value)
        if not text:
            return []
        return [
            normalize_space(part).lower()
            for part in _re.split(r"\s*\|\s*|\s*;\s*|\s*,\s*", text)
            if normalize_space(part)
        ]

    def title_overlap_ratio(query, title):
        query_tokens = set(tokenize(query))
        title_tokens = {token for token in tokenize(title) if len(token) >= 4}
        if not query_tokens or not title_tokens:
            return 0.0
        return len(query_tokens & title_tokens) / len(query_tokens)

    def distinctive_metadata_terms(value):
        terms = []
        for phrase in split_metadata_values(value):
            phrase_tokens = tokenize(phrase)
            if len(phrase_tokens) >= 2:
                terms.append(" ".join(phrase_tokens))
            terms.extend(
                token for token in phrase_tokens
                if len(token) >= 3 and token not in METADATA_GENERIC_TOKENS
            )
        return terms

    def diagnostic_metadata_terms(value):
        terms = []
        for phrase in split_metadata_values(value):
            phrase_tokens = tokenize(phrase)
            if phrase_tokens:
                terms.append(" ".join(phrase_tokens))
            terms.extend(token for token in phrase_tokens if len(token) >= 3)
        return terms

    def metadata_term_groups(row):
        name_terms = []
        for column in METADATA_NAME_COLUMNS:
            name_terms.extend(distinctive_metadata_terms(row.get(column, "")))
        identifier_terms = []
        for column in METADATA_DIAGNOSTIC_COLUMNS:
            identifier_terms.extend(diagnostic_metadata_terms(row.get(column, "")))
        title = normalize_query_text(row.get("itemctx_title", ""))
        return {
            "name_terms": list(dict.fromkeys(name_terms)),
            "identifier_terms": list(dict.fromkeys(identifier_terms)),
            "title": title,
        }

    def scrub_target_metadata_cues(text, row):
        scrubbed = normalize_query_text(text)
        removed = []
        for label, pattern in [
            ("asin_pattern", ASIN_PATTERN),
            ("package_or_dosage_pattern", PACKAGE_PATTERN),
            ("seller_or_manufacturer_pattern", SELLER_PATTERN),
        ]:
            updated = normalize_query_text(pattern.sub(" ", scrubbed))
            if updated != scrubbed:
                removed.append(label)
            scrubbed = updated
        groups = metadata_term_groups(row)
        for term in groups["name_terms"] + groups["identifier_terms"]:
            updated = remove_exact_phrase(scrubbed, term)
            if updated != scrubbed:
                removed.append(term)
            scrubbed = updated
        title = groups["title"]
        if token_count(title) >= 2:
            updated = remove_exact_phrase(scrubbed, title)
            if updated != scrubbed:
                removed.append("exact_title_phrase")
            scrubbed = updated
        return normalize_query_text(scrubbed), " | ".join(dict.fromkeys(removed))

    def metadata_leakage_flags(text, row):
        normalized = normalize_query_text(text)
        groups = metadata_term_groups(row)
        title = groups["title"]
        return {
            "brand_or_name_leak_flag": int(any(
                exact_phrase_present(normalized, term) for term in groups["name_terms"]
            )),
            "identifier_like_leak_flag": int(any(
                exact_phrase_present(normalized, term) for term in groups["identifier_terms"]
            )),
            "asin_leak_flag": int(bool(ASIN_PATTERN.search(normalized))),
            "package_cue_flag": int(bool(PACKAGE_PATTERN.search(normalized))),
            "seller_manufacturer_leak_flag": int(bool(SELLER_PATTERN.search(normalized))),
            "exact_title_phrase_flag": int(
                token_count(title) >= 2 and exact_phrase_present(normalized, title)
            ),
            "title_overlap_ratio": float(title_overlap_ratio(normalized, title)),
        }

    BLOCKING_FLAGS = [
        "brand_or_name_leak_flag",
        "identifier_like_leak_flag",
        "asin_leak_flag",
        "package_cue_flag",
        "seller_manufacturer_leak_flag",
        "exact_title_phrase_flag",
    ]

    def blocking_leakage_count(flags):
        return sum(flags[column] for column in BLOCKING_FLAGS)

    def truncate_tokens(text, max_tokens):
        tokens = normalize_query_text(text).split()
        return " ".join(tokens[:max_tokens])

    return types.SimpleNamespace(
        normalize_query_text=normalize_query_text,
        tokenize=tokenize,
        token_count=token_count,
        scrub=scrub_target_metadata_cues,
        flags=metadata_leakage_flags,
        blocking_flags=BLOCKING_FLAGS,
        blocking_count=blocking_leakage_count,
        truncate_tokens=truncate_tokens,
    )


SCRUB = _build_scrub_namespace()

# Isolation audit: the transplanted Notebook 07 helper must survive at module scope.
_probe = "  Fine   Lines  "
assert SCRUB.normalize_query_text(_probe) == "fine lines", "Notebook 06 scrub namespace is broken."
print("Scrub namespace built. Blocking flags:", len(SCRUB.blocking_flags))


Scrub namespace built. Blocking flags: 6


In [6]:
# Extend the production validation checks for the appendix comparison.
# =========================================================
# Inputs and Validation
# =========================================================
load_start = time.perf_counter()

required_paths = [
    QUERY_CACHE_PATH,
    QUERY_CONTRACT_PATH,
    ITEM_DOCS_PATH,
    ITEM_FACETS_PATH,
    GRAPH_EDGES_PATH,
    ITEM_SCHEMA_PATH,
    SAMPLED_USERS_PATH,
    FINAL_SAMPLING_POOL_PATH,
    RETRIEVAL_ARTIFACT_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required inputs: {missing_paths}")

query_contract = load_json(QUERY_CONTRACT_PATH)
retrieval_manifest = load_json(RETRIEVAL_ARTIFACT_MANIFEST_PATH)
validate_query_contract(query_contract)
validate_retrieval_manifest(retrieval_manifest)

query_schema_columns = pq.ParquetFile(QUERY_CACHE_PATH).schema.names
query_alias_columns = sorted(KNOWN_QUERY_ALIAS_COLUMNS.intersection(query_schema_columns))
expected_query_alias_columns = [] if COMPATIBILITY_QUERY_ALIAS is None else [COMPATIBILITY_QUERY_ALIAS]
if query_alias_columns != expected_query_alias_columns:
    raise RuntimeError(
        "Active query aliases are ambiguous: "
        f"expected {expected_query_alias_columns}, found {query_alias_columns}."
    )
required_query_columns = [
    "case_id",
    "user_id",
    "regime",
    "target_parent_asin",
    "target_selection_mode",
    "target_timestamp_ms",
    ACTIVE_QUERY_COLUMN,
    "query_evidence_source",
    "target_metadata_fallback_used",
    "item_context_fallback_used",
    "item_metadata_evidence_used",
    "historical_review_evidence_used",
    "user_prior_evidence_used",
    "rating_evidence_used",
    "sentiment_evidence_used",
    "insufficient_review_evidence",
    "query_generation_status",
    "query_specific_facet_family_count",
    "query_generic_anchor_terms",
    "query_generic_utility_terms",
    "query_generic_anchor_rate",
    "query_specific_facet_cue_rate",
    "query_clean_is_active",
    *QUERY_PASSTHROUGH_COLUMNS,
]
if COMPATIBILITY_QUERY_ALIAS is not None:
    required_query_columns.append(COMPATIBILITY_QUERY_ALIAS)
missing_query_columns = [column for column in required_query_columns if column not in query_schema_columns]
if missing_query_columns:
    raise RuntimeError(f"Query cache is missing required columns: {missing_query_columns}")
query_load_columns = required_query_columns + [
    column for column in OPTIONAL_QUERY_AUDIT_COLUMNS
    if column in query_schema_columns and column not in required_query_columns
]
query_load_columns = list(dict.fromkeys(query_load_columns))
queries_full = pd.read_parquet(QUERY_CACHE_PATH, columns=query_load_columns).copy()
sampled_rank_df = pd.read_parquet(
    SAMPLED_USERS_PATH,
    columns=["case_id", "target_rank_desc"],
).copy()

eligible_rank_df = pd.read_parquet(
    FINAL_SAMPLING_POOL_PATH,
    columns=["case_id", "target_rank_desc"],
).copy()

target_rank_lookup_df = (
    pd.concat([sampled_rank_df, eligible_rank_df], ignore_index=True)
    .dropna(subset=["case_id"])
    .copy()
)
target_rank_lookup_df["case_id"] = target_rank_lookup_df["case_id"].astype(str).str.strip()
target_rank_lookup_df["target_rank_desc"] = pd.to_numeric(
    target_rank_lookup_df["target_rank_desc"],
    errors="coerce",
)
target_rank_lookup_df = (
    target_rank_lookup_df
    .dropna(subset=["target_rank_desc"])
    .drop_duplicates("case_id", keep="first")
)

queries_full["case_id"] = queries_full["case_id"].astype(str).str.strip()
queries_full = queries_full.merge(
    target_rank_lookup_df,
    on="case_id",
    how="left",
    validate="many_to_one",
)

if queries_full["target_rank_desc"].isna().any():
    missing_rank_cases = queries_full.loc[
        queries_full["target_rank_desc"].isna(),
        ["case_id", "replacement_case_used", "replacement_source_case_id"],
    ].head(20)
    display(missing_rank_cases)
    raise RuntimeError("target_rank_desc must be non-null after sampled + eligible pool merge.")

item_docs = pd.read_parquet(
    ITEM_DOCS_PATH,
    columns=[ITEM_ID_COLUMN, DENSE_TEXT_COLUMN, SPARSE_TEXT_COLUMN, BRAND_TEXT_COLUMN],
).copy()
facet_flag_columns = [
    FACET_ITEM_ID_COLUMN,
    "source_column",
    "facet_role",
    *GRAPH_FACET_FLAG_COLUMNS,
]
facet_value_columns = [
    FACET_ITEM_ID_COLUMN,
    FACET_VALUE_COLUMN,
    "source_column",
    "facet_role",
    *GRAPH_FACET_FLAG_COLUMNS,
]
facet_schema_columns = pq.ParquetFile(ITEM_FACETS_PATH).schema.names
missing_facet_columns = [
    column for column in facet_value_columns
    if column not in facet_schema_columns
]
if missing_facet_columns:
    raise RuntimeError(f"Item facets are missing required columns: {missing_facet_columns}")
item_facet_flags = pd.read_parquet(
    ITEM_FACETS_PATH,
    columns=facet_flag_columns,
).copy()
facet_filter_audit_df = audit_global_review_facet_flags(item_facet_flags)
metadata_item_facets_raw = pq.read_table(
    ITEM_FACETS_PATH,
    columns=facet_value_columns,
    filters=[("is_core_graph_facet", "=", True)],
).to_pandas()
brand_item_facets_raw = pq.read_table(
    ITEM_FACETS_PATH,
    columns=facet_value_columns,
    filters=[("is_brand_graph_facet", "=", True)],
).to_pandas()
review_item_facets_raw = pq.read_table(
    ITEM_FACETS_PATH,
    columns=facet_value_columns,
    filters=[
        ("is_review_derived", "=", True),
        ("is_global_review_graph_facet", "=", True),
    ],
).to_pandas()
graph_edge_columns = [
    "source_node_type",
    "target_node_type",
    "source_column",
    "facet_role",
    *GRAPH_FACET_FLAG_COLUMNS,
]
graph_edges = pd.read_parquet(GRAPH_EDGES_PATH, columns=graph_edge_columns).copy()
validate_global_review_graph_edges(graph_edges)

EXPECTED_ROWS = int(len(queries_full))
for column in ["case_id", "user_id", "regime", "target_parent_asin"]:
    if queries_full[column].isna().any() or queries_full[column].map(normalize_space).eq("").any():
        raise RuntimeError(f"{column} must be non-null and non-empty.")
if queries_full["case_id"].astype(str).duplicated().any():
    raise RuntimeError("case_id must be unique.")
if queries_full["user_id"].astype(str).duplicated().any():
    raise RuntimeError("user_id must be unique.")
if queries_full["target_timestamp_ms"].isna().any():
    raise RuntimeError("target_timestamp_ms must be non-null.")
if not queries_full["target_selection_mode"].fillna("").astype(str).eq(EXPECTED_TARGET_SELECTION_MODE).all():
    observed = sorted(queries_full["target_selection_mode"].astype(str).unique().tolist())
    raise RuntimeError(f"Unexpected target_selection_mode values: {observed}")
if MAX_TARGET_RANK_ALLOWED is not None:
    if queries_full["target_rank_desc"].isna().any():
        raise RuntimeError("target_rank_desc must be non-null.")
    if queries_full["target_rank_desc"].astype(int).gt(MAX_TARGET_RANK_ALLOWED).any():
        raise RuntimeError(f"target_rank_desc must not exceed {MAX_TARGET_RANK_ALLOWED}.")

observed_regime_counts = (
    queries_full["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
contract_target_per_regime = int(query_contract.get("target_per_regime", -1))
expected_contract_counts = {regime: int(contract_target_per_regime) for regime in REGIME_ORDER}
if observed_regime_counts != expected_contract_counts:
    raise RuntimeError(
        f"Regime count mismatch against Notebook 06 contract: actual {observed_regime_counts}, "
        f"expected {expected_contract_counts}"
    )
EXPECTED_REGIME_COUNTS = dict(observed_regime_counts)

active_query = queries_full[ACTIVE_QUERY_COLUMN].map(normalize_space)
if active_query.eq("").any():
    raise RuntimeError("The active query must be non-empty for every row.")
if COMPATIBILITY_QUERY_ALIAS is not None:
    compatibility_query = queries_full[COMPATIBILITY_QUERY_ALIAS].map(normalize_space)
    if not compatibility_query.eq(active_query).all():
        raise RuntimeError(f"{COMPATIBILITY_QUERY_ALIAS} must be an exact alias of query.")
if boolean_series(queries_full["query_clean_is_active"]).any():
    raise RuntimeError("query_clean must remain inactive for the production retrieval run.")
if not queries_full["query_evidence_source"].fillna("").astype(str).eq("target_review_safe_signals_only").all():
    raise RuntimeError("Every query must use target_review_safe_signals_only evidence.")
validate_false_columns(
    queries_full,
    [
        "target_metadata_fallback_used",
        "item_context_fallback_used",
        "item_metadata_evidence_used",
        "historical_review_evidence_used",
        "user_prior_evidence_used",
        "rating_evidence_used",
        "sentiment_evidence_used",
        "insufficient_review_evidence",
    ],
    "query cache",
)
if queries_full["query_specific_facet_family_count"].fillna(0).astype(int).le(0).any():
    raise RuntimeError("Every final query must retain at least one specific review-derived cue family.")
if queries_full["query_generation_status"].isna().any() or queries_full["query_generation_status"].map(normalize_space).eq("").any():
    raise RuntimeError("query_generation_status must be non-empty for every row.")

forbidden_loaded_query_columns = {
    "raw_review_text",
    "target_review_text",
    "review_title",
    "review_body",
    "brand",
    "title",
    "manufacturer",
    "seller",
    "query_safe_facet_text",
    "common_functional_facet_text",
    "historical_review_reputation_text",
    "review_reputation_facet_text",
    "prior_review_text",
    "prior_history_n",
}
loaded_forbidden = sorted(forbidden_loaded_query_columns.intersection(queries_full.columns))
if loaded_forbidden:
    raise RuntimeError(f"Prohibited query evidence columns were loaded: {loaded_forbidden}")

if item_docs[ITEM_ID_COLUMN].isna().any() or item_docs[ITEM_ID_COLUMN].map(normalize_space).eq("").any():
    raise RuntimeError("Item docs must contain non-null, non-empty parent_asin values.")
item_docs[ITEM_ID_COLUMN] = item_docs[ITEM_ID_COLUMN].astype(str)
if item_docs[ITEM_ID_COLUMN].duplicated().any():
    raise RuntimeError("Item docs must contain one row per parent_asin.")
if len(item_docs) == 0:
    raise RuntimeError("Global Review catalog must contain at least one item.")
for column in [DENSE_TEXT_COLUMN, SPARSE_TEXT_COLUMN]:
    if item_docs[column].fillna("").astype(str).str.strip().eq("").any():
        raise RuntimeError(f"{column} must be non-empty for every item.")
item_docs[BRAND_TEXT_COLUMN] = item_docs[BRAND_TEXT_COLUMN].fillna("").astype(str).map(normalize_space)
if item_docs[BRAND_TEXT_COLUMN].eq("").all():
    raise RuntimeError("The Global Review catalog must retain non-empty brand facets.")

item_docs = item_docs.sort_values(ITEM_ID_COLUMN, kind="stable").reset_index(drop=True)
catalog_size = int(len(item_docs))
EXPECTED_CANDIDATE_K = min(MAX_RETRIEVAL_K, catalog_size)
if EXPECTED_CANDIDATE_K <= 0:
    raise RuntimeError("Exact-K candidate budget must be positive.")
item_ids = item_docs[ITEM_ID_COLUMN].to_numpy(dtype=object)
item_brand_texts = item_docs[BRAND_TEXT_COLUMN].to_numpy(dtype=object)
item_id_set = set(item_ids)
item_to_position = {item_id: position for position, item_id in enumerate(item_ids)}
target_positions = queries_full["target_parent_asin"].astype(str).map(item_to_position)
if target_positions.isna().any():
    raise RuntimeError("At least one target item is missing from the Global Review catalog.")
queries_full["target_item_position"] = target_positions.astype(np.int32)

core_facets = canonicalize_graph_facets(metadata_item_facets_raw, item_to_position, "metadata")
brand_facets = canonicalize_graph_facets(brand_item_facets_raw, item_to_position, "brand")
review_facets = canonicalize_graph_facets(review_item_facets_raw, item_to_position, "review")
facet_filter_audit_df["metadata_functional_rows"] = int(len(core_facets))
facet_filter_audit_df["metadata_facet_items"] = int(core_facets["item_position"].nunique())
facet_filter_audit_df["brand_rows_used"] = int(len(brand_facets))
facet_filter_audit_df["brand_facet_items"] = int(brand_facets["item_position"].nunique())
facet_filter_audit_df["review_derived_rows"] = int(len(review_facets))
facet_filter_audit_df["review_facet_items"] = int(review_facets["item_position"].nunique())
facet_filter_audit_df["metadata_graph_edges"] = int(
    graph_edges["is_core_graph_facet"].fillna(False).astype(bool).sum()
)
facet_filter_audit_df["brand_graph_edges"] = int(
    graph_edges["is_brand_graph_facet"].fillna(False).astype(bool).sum()
)
facet_filter_audit_df["review_reputation_graph_edges"] = int(
    graph_edges["is_review_derived"].fillna(False).astype(bool).sum()
)

queries_full["active_query_text"] = active_query
queries_full["query_text_retrieval"] = queries_full["active_query_text"]
queries_full["query_text_graph"] = queries_full["active_query_text"].map(select_graph_query_text)
if queries_full["query_text_retrieval"].map(canonical_tokens).map(len).eq(0).any():
    raise RuntimeError("Dense/BM25 query text must contain at least one token.")
if queries_full["query_text_graph"].map(canonical_tokens).map(len).eq(0).any():
    raise RuntimeError("A final strict query produced no graph-safe tokens.")
if not queries_full["query_text_retrieval"].eq(queries_full["active_query_text"]).all():
    raise RuntimeError("Dense and BM25 must use the exact active Notebook 06 query.")
queries_full["query_method"] = QUERY_METHOD_LABEL

if RUN_SMOKE_TEST:
    if SMOKE_TEST_N > EXPECTED_ROWS:
        raise RuntimeError("SMOKE_TEST_N cannot exceed EXPECTED_ROWS.")
    evaluation_queries_df = (
        queries_full.sample(n=SMOKE_TEST_N, random_state=RANDOM_SEED)
        .sort_values(["regime", "case_id"], kind="stable")
        .reset_index(drop=True)
    )
else:
    evaluation_queries_df = queries_full.copy().reset_index(drop=True)

load_validation_runtime_sec = time.perf_counter() - load_start
print("Rows: queries", len(evaluation_queries_df))
print("Rows: items", len(item_docs))
print("Rows: metadata facets", len(core_facets))
print("Rows: brand facets", len(brand_facets))
print("Rows: historical review facets", len(review_facets))
print("Rows: Global Review graph edges", len(graph_edges))
print("Validation: inputs passed")


Rows: queries 2288
Rows: items 77502
Rows: metadata facets 464188
Rows: brand facets 74751
Rows: historical review facets 277049
Rows: Global Review graph edges 977717
Validation: inputs passed


In [7]:
# =========================================================
# Query Variant Construction (A, B, C, D)
# =========================================================
# A and C are read from the sealed Notebook 06 cache and are never regenerated:
# the appendix must compare against the queries the thesis actually used, and the
# rewrite endpoint is not bit-reproducible. B and D are generated here from the
# same query-safe evidence, then passed through the identical direct-cue scrub so
# that any HitRate difference is attributable to lexical freedom and not to a
# weaker leakage control.
variant_build_start = time.perf_counter()

if "query_seed" not in evaluation_queries_df.columns:
    raise RuntimeError(
        "Variant A requires the deterministic seed. The sealed query cache does not expose "
        "'query_seed'; regenerate the cache column list or supply the Notebook 06 seed export."
    )

# ---- Target-metadata negative dictionary (audit only; never a query input) ---
item_context_cols = ["parent_asin", "title", "facet_brand_text"]
# NOTE: pd.read_parquet(path, columns=[]) reads ZERO columns, so it cannot be
# used for schema introspection — it silently reports an empty column set and
# would drop negative-dictionary fields. Read the logical Arrow schema instead
# (pq.read_schema, not ParquetFile.schema, whose .names returns physical leaf
# names such as "element" for list-typed columns).
schema_cols = set(pq.read_schema(ITEM_SCHEMA_PATH).names)
if "identifier_diagnostic_text" in schema_cols:
    item_context_cols.append("identifier_diagnostic_text")
item_context_df = pd.read_parquet(ITEM_SCHEMA_PATH, columns=item_context_cols)
if "identifier_diagnostic_text" not in item_context_df.columns:
    item_context_df["identifier_diagnostic_text"] = ""
item_context_df["parent_asin"] = item_context_df["parent_asin"].astype(str).str.strip()
if item_context_df["parent_asin"].duplicated().any():
    raise RuntimeError("Item schema audit keys must be unique.")
item_context_df = item_context_df.rename(columns={
    "parent_asin": "target_parent_asin",
    "title": "itemctx_title",
    "facet_brand_text": "itemctx_facet_brand_text",
    "identifier_diagnostic_text": "itemctx_identifier_diagnostic_text",
})

base_df = evaluation_queries_df[[
    "case_id", "user_id", "regime", "target_parent_asin", "target_item_position",
    "active_query_text", "query_seed",
]].copy()
base_df["target_parent_asin"] = base_df["target_parent_asin"].astype(str).str.strip()
base_df = base_df.merge(item_context_df, on="target_parent_asin", how="left", validate="many_to_one")
for column in ["itemctx_title", "itemctx_facet_brand_text", "itemctx_identifier_diagnostic_text"]:
    base_df[column] = base_df[column].fillna("").astype(str)

base_df["variant_A_source_text"] = base_df["query_seed"].map(SCRUB.normalize_query_text)
base_df["variant_C_source_text"] = base_df["active_query_text"].map(SCRUB.normalize_query_text)
if base_df["variant_A_source_text"].eq("").any():
    raise RuntimeError("Variant A is empty for at least one case; the sealed seed is unusable.")
if base_df["variant_C_source_text"].eq("").any():
    raise RuntimeError("Variant C is empty for at least one case.")

ac_identical = int(base_df["variant_A_source_text"].eq(base_df["variant_C_source_text"]).sum())
print(f"A and C are string-identical for {ac_identical} of {len(base_df)} cases "
      f"({ac_identical / len(base_df):.1%}) — the accepted-rewrite subset is the remainder.")

# ---- Generation programs for B and D ----------------------------------------
import dspy


class LlmRewrite(dspy.Signature):
    """Rewrite the shopping need as one natural search query.

    Keep the same need. You may choose different wording and add ordinary product
    vocabulary. Do not invent a brand, a product name, a seller, a size or a
    package quantity. Return the query only.
    """
    seed_query = dspy.InputField(desc="deterministic review-safe need statement")
    query = dspy.OutputField(desc="one natural-language search query, 5 to 30 words")


class HydeExpansion(dspy.Signature):
    """Write a short product passage that would fully satisfy this shopping need.

    Two or three sentences describing the kind of product that answers the need,
    in the register of a retail product description. Do not invent a brand, a
    product name, a seller, a size or a package quantity. Return the passage only.
    """
    query = dspy.InputField(desc="retrieval query")
    passage = dspy.OutputField(desc="hypothetical product passage, 20 to 80 words")


def _configure_lm(max_tokens):
    from google.colab import userdata
    api_key = userdata.get(DEEPSEEK_COLAB_SECRET)
    if not api_key:
        if FAIL_IF_LLM_UNAVAILABLE:
            raise RuntimeError(f"DeepSeek API key required in Colab secret {DEEPSEEK_COLAB_SECRET!r}.")
        return None
    return dspy.LM(
        model=DEEPSEEK_MODEL,
        api_key=api_key,
        api_base=DEEPSEEK_BASE_URL,
        temperature=DSPY_TEMPERATURE,
        max_tokens=max_tokens,
    )


def _call_program(program, lm, field_name, **kwargs):
    for attempt in range(1, DSPY_MAX_RETRIES + 1):
        try:
            with dspy.context(lm=lm):
                prediction = program(**kwargs)
            text = SCRUB.normalize_query_text(getattr(prediction, field_name, ""))
            if text:
                return text, "generated"
        except Exception as error:  # noqa: BLE001 - recorded, then retried or failed soft
            if attempt == DSPY_MAX_RETRIES:
                return "", f"error:{type(error).__name__}"
        if attempt < DSPY_MAX_RETRIES:
            time.sleep(DSPY_RETRY_SLEEP_SECONDS)
    return "", "empty_output"


# ---- Generate (or reload from the disposable cache) --------------------------
generation_runtime_sec = {"B": 0.0, "D": 0.0}
cached_df = None
if GENERATION_CACHE_PATH.exists():
    cached_df = pd.read_parquet(GENERATION_CACHE_PATH)
    cached_df["case_id"] = cached_df["case_id"].astype(str)
    required_cached = {"case_id", "variant_B_raw_text", "variant_D_raw_text"}
    if not required_cached.issubset(cached_df.columns):
        cached_df = None
    elif not set(base_df["case_id"]).issubset(set(cached_df["case_id"])):
        cached_df = None

if cached_df is not None:
    print("Generation cache hit — no API calls will be made.")
    base_df = base_df.merge(
        cached_df[["case_id", "variant_B_raw_text", "variant_D_raw_text",
                   "variant_B_status", "variant_D_status"]],
        on="case_id", how="left", validate="one_to_one",
    )
else:
    if not USE_LLM_GENERATION:
        raise RuntimeError("No generation cache and USE_LLM_GENERATION is False.")
    lm_b = _configure_lm(DSPY_MAX_TOKENS["B"])
    lm_d = _configure_lm(DSPY_MAX_TOKENS["D"])
    rewrite_program = dspy.Predict(LlmRewrite)
    hyde_program = dspy.Predict(HydeExpansion)

    b_texts, b_status, d_texts, d_status = [], [], [], []
    for row in tqdm(base_df.itertuples(index=False), total=len(base_df), desc="B/D generation"):
        start = time.perf_counter()
        text_b, status_b = _call_program(rewrite_program, lm_b, "query", seed_query=row.variant_A_source_text)
        generation_runtime_sec["B"] += time.perf_counter() - start

        start = time.perf_counter()
        text_d, status_d = _call_program(hyde_program, lm_d, "passage", query=row.variant_C_source_text)
        generation_runtime_sec["D"] += time.perf_counter() - start

        b_texts.append(text_b); b_status.append(status_b)
        d_texts.append(text_d); d_status.append(status_d)

    base_df["variant_B_raw_text"] = b_texts
    base_df["variant_B_status"] = b_status
    base_df["variant_D_raw_text"] = d_texts
    base_df["variant_D_status"] = d_status
    base_df[["case_id", "variant_B_raw_text", "variant_B_status",
             "variant_D_raw_text", "variant_D_status"]].to_parquet(GENERATION_CACHE_PATH, index=False)
    print("Generation cache written:", GENERATION_CACHE_PATH)

# ---- Scrub, cap, fall back, and audit ---------------------------------------
variant_rows = []
leak_rows = []
for row in base_df.itertuples(index=False):
    row_dict = row._asdict()
    raw_by_variant = {
        "A": (row_dict["variant_A_source_text"], "sealed", D_MAX_TOKENS),
        "B": (row_dict["variant_B_raw_text"], row_dict["variant_B_status"], B_MAX_TOKENS),
        "C": (row_dict["variant_C_source_text"], "sealed", D_MAX_TOKENS),
        "D": (row_dict["variant_D_raw_text"], row_dict["variant_D_status"], D_MAX_TOKENS),
    }
    for variant_key in VARIANT_KEYS:
        raw_text, status, cap = raw_by_variant[variant_key]
        raw_text = SCRUB.normalize_query_text(raw_text or "")
        pre_flags = SCRUB.flags(raw_text, row_dict) if raw_text else {
            name: 0 for name in SCRUB.blocking_flags
        } | {"title_overlap_ratio": 0.0}
        pre_blocking = SCRUB.blocking_count(pre_flags) if raw_text else 0

        scrubbed, removed_terms = SCRUB.scrub(raw_text, row_dict) if raw_text else ("", "")
        scrubbed = SCRUB.truncate_tokens(scrubbed, cap)

        fallback_used = False
        if SCRUB.token_count(scrubbed) == 0:
            fallback_source = "C" if variant_key == "D" else "A"
            scrubbed = row_dict[f"variant_{fallback_source}_source_text"]
            fallback_used = True
            status = f"{status}|fallback_to_{fallback_source}"

        graph_text = select_graph_query_text(scrubbed)
        if not canonical_tokens(scrubbed):
            raise RuntimeError(f"Variant {variant_key} produced no retrieval token for {row_dict['case_id']}.")
        if not canonical_tokens(graph_text):
            graph_text = scrubbed  # graph stopword filter emptied the text; keep the retrieval form

        variant_rows.append({
            "case_id": str(row_dict["case_id"]),
            "user_id": str(row_dict["user_id"]),
            "regime": str(row_dict["regime"]),
            "target_parent_asin": str(row_dict["target_parent_asin"]),
            "target_item_position": int(row_dict["target_item_position"]),
            "query_variant": variant_key,
            "query_variant_label": VARIANT_LABELS[variant_key],
            "query_source": VARIANT_SOURCE[variant_key],
            "lexical_freedom": VARIANT_LEXICAL_FREEDOM[variant_key],
            "generation_status": status,
            "variant_fallback_used": bool(fallback_used),
            "query_text_retrieval": scrubbed,
            "query_text_graph": graph_text,
            "query_token_count": SCRUB.token_count(scrubbed),
        })
        leak_rows.append({
            "case_id": str(row_dict["case_id"]),
            "query_variant": variant_key,
            "pre_scrub_blocking_flag_count": int(pre_blocking),
            "pre_scrub_any_blocking_flag": int(pre_blocking > 0),
            "pre_scrub_title_overlap_ratio": float(pre_flags["title_overlap_ratio"]),
            "scrub_removed_terms": removed_terms,
            **{name: int(pre_flags[name]) for name in SCRUB.blocking_flags},
        })

variant_queries_df = pd.DataFrame(variant_rows)
variant_leakage_df = pd.DataFrame(leak_rows)
variant_build_runtime_sec = time.perf_counter() - variant_build_start

expected_variant_rows = len(base_df) * len(VARIANT_KEYS)
if len(variant_queries_df) != expected_variant_rows:
    raise RuntimeError(f"Expected {expected_variant_rows} variant rows, built {len(variant_queries_df)}.")

print("\nToken length by variant:")
display(
    variant_queries_df.groupby("query_variant")["query_token_count"]
    .agg(["mean", "median", "max"]).round(2)
)
print("\nPre-scrub blocking-flag rate by variant (share of cases whose generated text "
      "tripped the direct-cue audit before scrubbing):")
display(
    variant_leakage_df.groupby("query_variant")["pre_scrub_any_blocking_flag"]
    .mean().round(4).rename("pre_scrub_blocking_flag_rate")
)


A and C are string-identical for 2088 of 2288 cases (91.3%) — the accepted-rewrite subset is the remainder.


B/D generation:   0%|          | 0/2288 [00:00<?, ?it/s]

Generation cache written: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/_disposable_query_variant_comparison/variant_generated_queries_face.parquet

Token length by variant:


,mean,median,max
query_variant,,,
A,8.12,8.0,14
B,8.63,8.0,18
C,8.11,8.0,15
D,47.86,48.0,81



Pre-scrub blocking-flag rate by variant (share of cases whose generated text tripped the direct-cue audit before scrubbing):


,pre_scrub_blocking_flag_rate
query_variant,
A,0.0000
B,0.0057
C,0.0000
D,0.0726


In [8]:
# =========================================================
# Item-Side Retrieval Indexes (built once; shared by all variants)
# =========================================================
# Index construction is offline cost and is excluded from every reported
# per-query figure, exactly as in the sealed Notebook 07.
offline_index_start = time.perf_counter()

model = SentenceTransformer(EMBEDDING_MODEL_NAME)
item_embeddings = model.encode(
    item_docs[DENSE_TEXT_COLUMN].astype(str).tolist(),
    batch_size=ITEM_EMBEDDING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype(np.float32)
dense_index = faiss.IndexFlatIP(item_embeddings.shape[1])
dense_index.add(np.ascontiguousarray(item_embeddings))

sparse_corpus_tokens = [
    tokenize_sparse_document(text)
    for text in item_docs[SPARSE_TEXT_COLUMN].astype(str)
]
if any(len(tokens) == 0 for tokens in sparse_corpus_tokens):
    raise RuntimeError("Every production sparse item document must contain an indexed token.")
bm25 = BM25Okapi(sparse_corpus_tokens)

metadata_graph_index = build_graph_index(core_facets, len(item_docs))
review_reputation_graph_index = build_graph_index(review_facets, len(item_docs))
if not metadata_graph_index["token_items"] and not metadata_graph_index["phrase_items"]:
    raise RuntimeError("The metadata functional-facet graph index is empty.")
if not review_reputation_graph_index["token_items"] and not review_reputation_graph_index["phrase_items"]:
    raise RuntimeError("The historical review-derived graph index is empty.")

offline_index_runtime_sec = time.perf_counter() - offline_index_start
print("Rows: dense index", dense_index.ntotal)
print("Rows: metadata graph phrases/tokens",
      len(metadata_graph_index["phrase_items"]), "/", len(metadata_graph_index["token_items"]))
print("Rows: review graph phrases/tokens",
      len(review_reputation_graph_index["phrase_items"]), "/", len(review_reputation_graph_index["token_items"]))
print(f"Offline index construction: {offline_index_runtime_sec:.1f} s (excluded from all reported runtimes)")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/606 [00:00<?, ?it/s]

Rows: dense index 77502
Rows: metadata graph phrases/tokens 7000 / 4994
Rows: review graph phrases/tokens 14 / 47
Offline index construction: 9434.6 s (excluded from all reported runtimes)


In [9]:
# =========================================================
# Retrieval Evaluation — 4 Variants x 4 Methods at Exact K
# =========================================================
# Candidate pools are NOT exported. The target rank at depth K is sufficient for
# every reported quantity: HitRate at any k <= K is a threshold on that one rank,
# so the depth sweep costs no additional retrieval.
per_query_rows = []
variant_component_runtime = {}
variant_query_counts = {}

for variant_key in VARIANT_KEYS:
    subset = (
        variant_queries_df.loc[variant_queries_df["query_variant"] == variant_key]
        .sort_values("case_id", kind="stable")
        .reset_index(drop=True)
    )
    variant_query_counts[variant_key] = int(len(subset))
    component_runtime = defaultdict(float)

    embed_start = time.perf_counter()
    query_embeddings = model.encode(
        subset["query_text_retrieval"].astype(str).tolist(),
        batch_size=QUERY_EMBEDDING_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)
    component_runtime["query_embedding"] = time.perf_counter() - embed_start

    search_start = time.perf_counter()
    dense_scores, dense_positions = dense_index.search(
        np.ascontiguousarray(query_embeddings), EXPECTED_CANDIDATE_K
    )
    component_runtime["dense_faiss_search"] = time.perf_counter() - search_start
    if dense_positions.shape != (len(subset), EXPECTED_CANDIDATE_K):
        raise RuntimeError(f"Unexpected dense output shape for variant {variant_key}.")
    if (dense_positions < 0).any():
        raise RuntimeError(f"Dense search returned invalid positions for variant {variant_key}.")

    for query_index, query_row in enumerate(
        tqdm(subset.itertuples(index=False), total=len(subset), desc=f"variant {variant_key}")
    ):
        dense_items = dense_positions[query_index].astype(np.int32).tolist()
        dense_rank_map = rank_map(dense_items)

        bm25_start = time.perf_counter()
        bm25_query_tokens = tokenize_sparse_query(query_row.query_text_retrieval)
        bm25_selected, _ = bm25_top_exact(bm25, bm25_query_tokens, EXPECTED_CANDIDATE_K)
        bm25_items = bm25_selected.astype(np.int32).tolist()
        component_runtime["bm25_retrieval"] += time.perf_counter() - bm25_start
        bm25_rank_map = rank_map(bm25_items)

        hybrid_start = time.perf_counter()
        hybrid_items, _ = reciprocal_rank_fusion(
            {"dense": dense_rank_map, "bm25": bm25_rank_map}, HYBRID_WEIGHTS, EXPECTED_CANDIDATE_K
        )
        component_runtime["dense_bm25_rrf"] += time.perf_counter() - hybrid_start

        graph_start = time.perf_counter()
        metadata_graph_items = top_items_from_scores(
            graph_score_map(query_row.query_text_retrieval, query_row.query_text_graph, metadata_graph_index),
            EXPECTED_CANDIDATE_K,
        )
        component_runtime["metadata_graph_retrieval"] += time.perf_counter() - graph_start
        metadata_graph_rank_map = rank_map(metadata_graph_items)

        review_graph_start = time.perf_counter()
        review_graph_items = top_items_from_scores(
            graph_score_map(query_row.query_text_retrieval, query_row.query_text_graph, review_reputation_graph_index),
            MAX_RETRIEVAL_K,
        )
        component_runtime["review_graph_retrieval"] += time.perf_counter() - review_graph_start
        review_graph_rank_map = rank_map(review_graph_items)

        graph_hybrid_start = time.perf_counter()
        graph_hybrid_items, _ = reciprocal_rank_fusion(
            {
                "dense": dense_rank_map,
                "bm25": bm25_rank_map,
                "metadata_graph": metadata_graph_rank_map,
                "review_reputation_graph": review_graph_rank_map,
            },
            GRAPH_HYBRID_WEIGHTS,
            EXPECTED_CANDIDATE_K,
        )
        component_runtime["graph_hybrid_rrf"] += time.perf_counter() - graph_hybrid_start

        method_rankings = {
            "dense_bert": dense_items,
            "bm25": bm25_items,
            "hybrid_dense_bm25": hybrid_items,
            "graph_hybrid": graph_hybrid_items,
        }
        target_position = int(query_row.target_item_position)
        for method_key, ranked_items in method_rankings.items():
            if len(ranked_items) != EXPECTED_CANDIDATE_K or len(set(ranked_items)) != EXPECTED_CANDIDATE_K:
                raise RuntimeError(
                    f"{method_key} returned an invalid candidate list for "
                    f"variant {variant_key}, case_id={query_row.case_id}."
                )
            rank = target_rank(ranked_items, target_position)
            per_query_rows.append({
                "case_id": str(query_row.case_id),
                "user_id": str(query_row.user_id),
                "regime": str(query_row.regime),
                "query_variant": variant_key,
                "query_variant_label": VARIANT_LABELS[variant_key],
                "method_key": method_key,
                "retrieval_method": METHOD_DISPLAY_NAMES[method_key],
                "evidence_scope": EVIDENCE_SCOPE,
                "query_token_count": int(query_row.query_token_count),
                "target_rank": rank,
                **metric_record(rank),
            })

    variant_component_runtime[variant_key] = dict(component_runtime)
    total_sec = sum(component_runtime.values())
    print(f"variant {variant_key}: {len(subset)} queries, "
          f"{total_sec:.1f} s of instrumented retrieval "
          f"({1000.0 * total_sec / max(len(subset), 1):.1f} ms/query summed over components)")

variant_per_query_metrics_df = pd.DataFrame(per_query_rows)
expected_metric_rows = len(base_df) * len(VARIANT_KEYS) * len(METHOD_KEYS)
if len(variant_per_query_metrics_df) != expected_metric_rows:
    raise RuntimeError(
        f"Expected {expected_metric_rows} metric rows, produced {len(variant_per_query_metrics_df)}."
    )
print("\nRows: per-query metrics", len(variant_per_query_metrics_df))


variant A:   0%|          | 0/2288 [00:00<?, ?it/s]

variant A: 2288 queries, 1109.5 s of instrumented retrieval (484.9 ms/query summed over components)


variant B:   0%|          | 0/2288 [00:00<?, ?it/s]

variant B: 2288 queries, 839.0 s of instrumented retrieval (366.7 ms/query summed over components)


variant C:   0%|          | 0/2288 [00:00<?, ?it/s]

variant C: 2288 queries, 1054.0 s of instrumented retrieval (460.7 ms/query summed over components)


variant D:   0%|          | 0/2288 [00:00<?, ?it/s]

variant D: 2288 queries, 4526.3 s of instrumented retrieval (1978.3 ms/query summed over components)

Rows: per-query metrics 36608


In [10]:
# =========================================================
# Summaries, Runtime, Leakage Audit and Paired Inference
# =========================================================
# ---- 1. Figure data: HitRate by candidate pool depth -------------------------
pool_depth_df = pool_depth_summary(
    variant_per_query_metrics_df, ["query_variant", "query_variant_label", "method_key", "retrieval_method"]
)
figure_df = pool_depth_df.loc[pool_depth_df["candidate_pool_depth"].isin(FIGURE_DEPTHS)].copy()
figure_df = figure_df.sort_values(
    ["query_variant", "method_key", "candidate_pool_depth"], kind="stable"
).reset_index(drop=True)

headline_df = (
    figure_df.loc[figure_df["candidate_pool_depth"] == HEADLINE_DEPTH,
                  ["query_variant", "query_variant_label", "method_key", "retrieval_method",
                   "n_queries", "HitRate", "NDCG", "MRR"]]
    .rename(columns={"HitRate": f"HitRate@{HEADLINE_DEPTH}",
                     "NDCG": f"NDCG@{HEADLINE_DEPTH}",
                     "MRR": f"MRR@{HEADLINE_DEPTH}"})
    .sort_values(["method_key", "query_variant"], kind="stable")
    .reset_index(drop=True)
)
print(f"HitRate@{HEADLINE_DEPTH} by variant and method:")
display(headline_df.pivot(index="retrieval_method", columns="query_variant",
                          values=f"HitRate@{HEADLINE_DEPTH}").round(4))

# ---- 2. Runtime -------------------------------------------------------------
# These are amortized batch-throughput figures, not online latencies: query
# embedding and the FAISS search are executed once per variant over all queries
# and divided by the query count. Components shared by several methods are
# counted once in each row, so METHOD ROWS ARE NOT ADDITIVE. Feature assembly and
# index construction are excluded. Consistent with Section 6.5 of the thesis,
# these measurements support no cross-method serving-cost ranking and must not be
# compared with the differently scoped figures of Appendix 6.C.
runtime_component_rows = []
for variant_key, components in variant_component_runtime.items():
    n_queries = max(variant_query_counts[variant_key], 1)
    for component, seconds in components.items():
        runtime_component_rows.append({
            "query_variant": variant_key,
            "query_variant_label": VARIANT_LABELS[variant_key],
            "component": component,
            "total_sec": round(float(seconds), 6),
            "amortized_ms_per_query": round(1000.0 * float(seconds) / n_queries, 4),
            "n_queries": n_queries,
        })
runtime_components_df = pd.DataFrame(runtime_component_rows)

METHOD_COMPONENTS = {
    "dense_bert": ["query_embedding", "dense_faiss_search"],
    "bm25": ["bm25_retrieval"],
    "hybrid_dense_bm25": ["query_embedding", "dense_faiss_search", "bm25_retrieval", "dense_bm25_rrf"],
    "graph_hybrid": ["query_embedding", "dense_faiss_search", "bm25_retrieval",
                     "metadata_graph_retrieval", "review_graph_retrieval", "graph_hybrid_rrf"],
}
runtime_method_rows = []
for variant_key, components in variant_component_runtime.items():
    n_queries = max(variant_query_counts[variant_key], 1)
    for method_key, parts in METHOD_COMPONENTS.items():
        seconds = sum(float(components.get(part, 0.0)) for part in parts)
        runtime_method_rows.append({
            "query_variant": variant_key,
            "query_variant_label": VARIANT_LABELS[variant_key],
            "method_key": method_key,
            "retrieval_method": METHOD_DISPLAY_NAMES[method_key],
            "amortized_ms_per_query": round(1000.0 * seconds / n_queries, 4),
            "components_counted": " + ".join(parts),
            "measurement_scope": "amortized_batch_throughput_not_online_latency",
            "rows_additive": False,
        })
runtime_method_df = pd.DataFrame(runtime_method_rows)

baseline_runtime = (
    runtime_method_df.loc[runtime_method_df["query_variant"] == ADOPTED_VARIANT]
    .set_index("method_key")["amortized_ms_per_query"].to_dict()
)
runtime_method_df["relative_to_variant_C_same_method"] = runtime_method_df.apply(
    lambda r: round(r["amortized_ms_per_query"] / baseline_runtime[r["method_key"]], 3)
    if baseline_runtime.get(r["method_key"], 0) > 0 else np.nan,
    axis=1,
)
print("\nAmortized ms/query by variant and method (NOT online latency; rows not additive):")
display(runtime_method_df.pivot(index="retrieval_method", columns="query_variant",
                                values="amortized_ms_per_query").round(2))

# ---- 3. Leakage-flag audit --------------------------------------------------
leakage_summary_df = (
    variant_leakage_df.groupby("query_variant")
    .agg(
        n_cases=("case_id", "size"),
        pre_scrub_blocking_flag_rate=("pre_scrub_any_blocking_flag", "mean"),
        mean_pre_scrub_flag_count=("pre_scrub_blocking_flag_count", "mean"),
        mean_pre_scrub_title_overlap=("pre_scrub_title_overlap_ratio", "mean"),
        **{name: (name, "mean") for name in SCRUB.blocking_flags},
    )
    .reset_index()
)
leakage_summary_df["lexical_freedom"] = leakage_summary_df["query_variant"].map(VARIANT_LEXICAL_FREEDOM)
print("\nPre-scrub direct-cue audit (measured on the text as constructed, before scrubbing):")
display(leakage_summary_df[["query_variant", "lexical_freedom", "n_cases",
                            "pre_scrub_blocking_flag_rate", "mean_pre_scrub_title_overlap"]].round(4))

# ---- 4. Paired inference against the adopted variant C ----------------------
try:
    from scipy.stats import binomtest as _binomtest

    def _exact_two_sided(b, c):
        n = b + c
        return 1.0 if n == 0 else float(_binomtest(b, n, 0.5, alternative="two-sided").pvalue)
except Exception:  # pragma: no cover - scipy is present in Colab
    def _exact_two_sided(b, c):
        n = b + c
        if n == 0:
            return 1.0
        z = abs(b - c) / math.sqrt(n)
        return float(math.erfc(z / math.sqrt(2)))


def _regime_stratified_bootstrap(deltas, regimes, replicates, seed):
    """Percentile CI for the paired difference, resampled within history regime
    and recombined at equal regime weights (the benchmark is regime-balanced)."""
    rng = np.random.default_rng(seed)
    unique_regimes = sorted(set(regimes))
    stratum_draws = np.empty((len(unique_regimes), replicates), dtype=np.float64)
    for position, regime in enumerate(unique_regimes):
        stratum = deltas[regimes == regime]
        picks = rng.integers(0, stratum.size, size=(replicates, stratum.size))
        stratum_draws[position] = stratum[picks].mean(axis=1)
    draws = stratum_draws.mean(axis=0)
    return float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))


hit_column = f"HitRate@{HEADLINE_DEPTH}"
wide = variant_per_query_metrics_df.pivot_table(
    index=["case_id", "regime", "method_key"], columns="query_variant", values=hit_column
).reset_index()

inference_rows = []
for method_key in METHOD_KEYS:
    frame = wide.loc[wide["method_key"] == method_key].sort_values("case_id", kind="stable")
    reference = frame[ADOPTED_VARIANT].to_numpy(dtype=np.float64)
    regimes = frame["regime"].to_numpy()
    for variant_key in VARIANT_KEYS:
        if variant_key == ADOPTED_VARIANT:
            continue
        current = frame[variant_key].to_numpy(dtype=np.float64)
        deltas = current - reference
        gained = int(((current == 1) & (reference == 0)).sum())
        lost = int(((current == 0) & (reference == 1)).sum())
        ci_low, ci_high = _regime_stratified_bootstrap(deltas, regimes, BOOTSTRAP_REPLICATES, BOOTSTRAP_SEED)
        delta_pp = 100.0 * float(deltas.mean())
        verdict = ""
        if variant_key == "A":
            verdict = (
                "non_inferior_to_C"
                if 100.0 * ci_low > -NON_INFERIORITY_MARGIN_PP
                else "non_inferiority_not_established"
            )
        inference_rows.append({
            "method_key": method_key,
            "retrieval_method": METHOD_DISPLAY_NAMES[method_key],
            "contrast": f"{variant_key} - {ADOPTED_VARIANT}",
            "n_cases": int(len(frame)),
            f"{hit_column}_variant": round(float(current.mean()), 6),
            f"{hit_column}_reference": round(float(reference.mean()), 6),
            "delta_pp": round(delta_pp, 4),
            "ci95_low_pp": round(100.0 * ci_low, 4),
            "ci95_high_pp": round(100.0 * ci_high, 4),
            "discordant_gained": gained,
            "discordant_lost": lost,
            "mcnemar_exact_p": round(_exact_two_sided(gained, lost), 6),
            "non_inferiority_margin_pp": NON_INFERIORITY_MARGIN_PP if variant_key == "A" else np.nan,
            "non_inferiority_verdict": verdict,
        })
paired_inference_df = pd.DataFrame(inference_rows)
print(f"\nPaired inference against variant {ADOPTED_VARIANT} at depth {HEADLINE_DEPTH} "
      f"(regime-stratified bootstrap, {BOOTSTRAP_REPLICATES} replicates, seed {BOOTSTRAP_SEED}):")
display(paired_inference_df[["retrieval_method", "contrast", "delta_pp", "ci95_low_pp",
                             "ci95_high_pp", "discordant_gained", "discordant_lost",
                             "mcnemar_exact_p", "non_inferiority_verdict"]])

# ---- 5. Reproduction report for variant C -----------------------------------
reproduction_rows = []
if SEALED_PER_QUERY_METRICS_PATH.exists():
    sealed = pd.read_parquet(SEALED_PER_QUERY_METRICS_PATH, columns=["case_id", "method_key", hit_column])
    sealed["case_id"] = sealed["case_id"].astype(str)
    current_c = variant_per_query_metrics_df.loc[
        variant_per_query_metrics_df["query_variant"] == ADOPTED_VARIANT,
        ["case_id", "method_key", hit_column],
    ].copy()
    merged = current_c.merge(sealed, on=["case_id", "method_key"], how="inner", suffixes=("_new", "_sealed"))
    for method_key in METHOD_KEYS:
        block = merged.loc[merged["method_key"] == method_key]
        agreement = float((block[f"{hit_column}_new"] == block[f"{hit_column}_sealed"]).mean()) if len(block) else np.nan
        reproduction_rows.append({
            "method_key": method_key,
            "retrieval_method": METHOD_DISPLAY_NAMES[method_key],
            "gate_mode": REPRODUCTION_GATE_MODE,
            "n_compared": int(len(block)),
            "per_case_hit_agreement": round(agreement, 6) if len(block) else None,
            f"{hit_column}_new": round(float(block[f"{hit_column}_new"].mean()), 6) if len(block) else None,
            f"{hit_column}_sealed": round(float(block[f"{hit_column}_sealed"].mean()), 6) if len(block) else None,
            "status": "compared",
        })
else:
    for method_key in METHOD_KEYS:
        reproduction_rows.append({
            "method_key": method_key,
            "retrieval_method": METHOD_DISPLAY_NAMES[method_key],
            "gate_mode": REPRODUCTION_GATE_MODE,
            "n_compared": 0,
            "per_case_hit_agreement": None,
            f"{hit_column}_new": None,
            f"{hit_column}_sealed": None,
            "status": "reference_missing",
        })
reproduction_df = pd.DataFrame(reproduction_rows)
print("\nVariant C reproduction against the sealed Notebook 07 per-query metrics "
      "(reported, not blocking — the rewrite endpoint is not bit-reproducible):")
display(reproduction_df)
if REPRODUCTION_GATE_BLOCKING and reproduction_df["per_case_hit_agreement"].fillna(0).lt(1.0).any():
    raise RuntimeError("Variant C failed the blocking reproduction gate.")


HitRate@1000 by variant and method:


query_variant,A,B,C,D
retrieval_method,,,,
BM25,0.2268,0.2212,0.2273,0.2054
Dense BERT,0.1853,0.1801,0.1879,0.2002
Dense-BM25 Hybrid,0.2264,0.2207,0.2268,0.2203
Graph-Hybrid,0.2273,0.2212,0.2273,0.2386



Amortized ms/query by variant and method (NOT online latency; rows not additive):


query_variant,A,B,C,D
retrieval_method,,,,
BM25,385.92,278.62,360.22,1823.55
Dense BERT,10.52,11.04,10.35,41.47
Dense-BM25 Hybrid,400.33,293.63,374.83,1869.57
Graph-Hybrid,481.04,362.73,456.42,1973.71



Pre-scrub direct-cue audit (measured on the text as constructed, before scrubbing):


,query_variant,lexical_freedom,n_cases,pre_scrub_blocking_flag_rate,mean_pre_scrub_title_overlap
0,A,none,2288,0.0000,0.1718
1,B,free_short,2288,0.0057,0.2084
2,C,seed_concept_set,2288,0.0000,0.1732
3,D,free_paragraph,2288,0.0726,0.0598



Paired inference against variant C at depth 1000 (regime-stratified bootstrap, 10000 replicates, seed 42):


,retrieval_method,contrast,delta_pp,ci95_low_pp,ci95_high_pp,discordant_gained,discordant_lost,mcnemar_exact_p,non_inferiority_verdict
0,Dense BERT,A - C,-0.2622,-0.6119,0.0437,4,10,0.179565,non_inferior_to_C
1,Dense BERT,B - C,-0.7867,-1.9231,0.3497,77,95,0.194747,
2,Dense BERT,D - C,1.2238,-0.2622,2.7098,165,137,0.120117,
3,BM25,A - C,-0.0437,-0.3059,0.2185,4,5,1.000000,non_inferior_to_C
4,BM25,B - C,-0.6119,-1.9231,0.6993,110,124,0.395464,
5,BM25,D - C,-2.1853,-4.1521,-0.1311,257,307,0.038992,
6,Dense-BM25 Hybrid,A - C,-0.0437,-0.3497,0.2185,5,6,1.000000,non_inferior_to_C
7,Dense-BM25 Hybrid,B - C,-0.6119,-1.8357,0.6119,100,114,0.374225,
8,Dense-BM25 Hybrid,D - C,-0.6556,-2.5350,1.1801,231,246,0.521555,
9,Graph-Hybrid,A - C,0.0000,-0.2622,0.2622,5,5,1.000000,non_inferior_to_C



Variant C reproduction against the sealed Notebook 07 per-query metrics (reported, not blocking — the rewrite endpoint is not bit-reproducible):


,method_key,retrieval_method,gate_mode,n_compared,per_case_hit_agreement,HitRate@1000_new,HitRate@1000_sealed,status
0,dense_bert,Dense BERT,hit_indicator,2288,1.0,0.187937,0.187937,compared
1,bm25,BM25,hit_indicator,2288,1.0,0.227273,0.227273,compared
2,hybrid_dense_bm25,Dense-BM25 Hybrid,hit_indicator,2288,1.0,0.226836,0.226836,compared
3,graph_hybrid,Graph-Hybrid,hit_indicator,2288,1.0,0.227273,0.227273,compared


In [11]:
# =========================================================
# Outputs and Run Manifest
# =========================================================
variant_queries_df.drop(columns=["query_text_retrieval", "query_text_graph"]).to_csv(
    VARIANT_TEXT_AUDIT_PATH, index=False, encoding="utf-8-sig"
)
variant_leakage_df.to_csv(LEAKAGE_AUDIT_PATH, index=False, encoding="utf-8-sig")
variant_per_query_metrics_df.to_parquet(PER_QUERY_METRICS_PATH, index=False)
figure_df.to_csv(POOL_DEPTH_SUMMARY_PATH, index=False, encoding="utf-8-sig")
headline_df.to_csv(HEADLINE_SUMMARY_PATH, index=False, encoding="utf-8-sig")
runtime_components_df.to_csv(RUNTIME_COMPONENTS_PATH, index=False, encoding="utf-8-sig")
runtime_method_df.to_csv(RUNTIME_METHOD_PATH, index=False, encoding="utf-8-sig")
paired_inference_df.to_csv(PAIRED_INFERENCE_PATH, index=False, encoding="utf-8-sig")
reproduction_df.to_csv(REPRODUCTION_REPORT_PATH, index=False, encoding="utf-8-sig")

run_manifest = {
    "notebook_name": NOTEBOOK_NAME,
    "category_id": CATEGORY_ID,
    "category_label": CATEGORY_LABEL,
    "study_class": "appendix_only_disposable",
    "run_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "n_cases": int(len(base_df)),
    "n_variants": len(VARIANT_KEYS),
    "n_methods": len(METHOD_KEYS),
    "regime_counts": {str(k): int(v) for k, v in base_df["regime"].value_counts().items()},
    "variants": {
        key: {
            "label": VARIANT_LABELS[key],
            "source": VARIANT_SOURCE[key],
            "lexical_freedom": VARIANT_LEXICAL_FREEDOM[key],
            "token_cap": B_MAX_TOKENS if key == "B" else (D_MAX_TOKENS if key == "D" else None),
            "llm_max_tokens": DSPY_MAX_TOKENS.get(key),
        }
        for key in VARIANT_KEYS
    },
    "adopted_variant": ADOPTED_VARIANT,
    "predeclared_before_results": {
        "non_inferiority_margin_pp": NON_INFERIORITY_MARGIN_PP,
        "non_inferiority_contrast": f"A - {ADOPTED_VARIANT} on HitRate@{HEADLINE_DEPTH}",
        "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        "bootstrap_seed": BOOTSTRAP_SEED,
        "bootstrap_scheme": "regime_stratified_paired_percentile",
        "headline_depth": HEADLINE_DEPTH,
        "figure_depths": FIGURE_DEPTHS,
        "variant_D_token_cap": D_MAX_TOKENS,
    },
    "retrieval_contract": {
        "method_keys": METHOD_KEYS,
        "rrf_constant": RRF_K,
        "hybrid_weights": HYBRID_WEIGHTS,
        "graph_hybrid_weights": GRAPH_HYBRID_WEIGHTS,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "exact_candidate_k": int(EXPECTED_CANDIDATE_K),
        "catalog_size": int(catalog_size),
        "evidence_scope": EVIDENCE_SCOPE,
        "brand_query_matching_enabled": BRAND_QUERY_MATCHING_ENABLED,
        "transplanted_verbatim_from": "07_query_retrieval_selection_face.ipynb (helper and input cells)",
    },
    "generation": {
        "model": DEEPSEEK_MODEL,
        "endpoint": DEEPSEEK_BASE_URL,
        "secret_name": DEEPSEEK_COLAB_SECRET,
        "temperature": DSPY_TEMPERATURE,
        "max_retries": DSPY_MAX_RETRIES,
        "generated_variants": GENERATED_VARIANTS,
        "generation_runtime_sec": {k: round(v, 3) for k, v in generation_runtime_sec.items()},
        "cache_path": str(GENERATION_CACHE_PATH),
    },
    "runtime": {
        "offline_index_construction_sec": round(offline_index_runtime_sec, 3),
        "variant_build_sec": round(variant_build_runtime_sec, 3),
        "measurement_scope": "amortized_batch_throughput_not_online_latency",
        "method_rows_additive": False,
        "excluded": ["offline_index_construction", "feature_assembly", "candidate_export"],
        "comparable_with_appendix_6C": False,
    },
    # NOTE: itertuples() renames non-identifier columns such as "HitRate@1000"
    # to positional placeholders, so the headline map is built by column access.
    "headline_hitrate": {
        f"{variant}|{method}": round(float(value), 6)
        for variant, method, value in zip(
            headline_df["query_variant"],
            headline_df["method_key"],
            headline_df[f"HitRate@{HEADLINE_DEPTH}"],
        )
    },
    "pre_scrub_blocking_flag_rate": {
        str(variant): round(float(rate), 6)
        for variant, rate in zip(
            leakage_summary_df["query_variant"],
            leakage_summary_df["pre_scrub_blocking_flag_rate"],
        )
    },
    "a_c_string_identical_cases": int(ac_identical),
    "sealed_artifact_writes": 0,
    "candidate_pools_exported": 0,
    "inputs": {
        "query_cache": str(QUERY_CACHE_PATH),
        "query_contract": str(QUERY_CONTRACT_PATH),
        "item_docs": str(ITEM_DOCS_PATH),
        "item_facets": str(ITEM_FACETS_PATH),
        "graph_edges": str(GRAPH_EDGES_PATH),
        "item_schema": str(ITEM_SCHEMA_PATH),
        "sampled_users": str(SAMPLED_USERS_PATH),
        "final_sampling_pool": str(FINAL_SAMPLING_POOL_PATH),
        "retrieval_artifact_manifest": str(RETRIEVAL_ARTIFACT_MANIFEST_PATH),
        "sealed_per_query_metrics_reference": str(SEALED_PER_QUERY_METRICS_PATH),
    },
    "outputs": {path.name: str(path) for path in EMITTED_PATHS},
}
with open(RUN_MANIFEST_PATH, "w", encoding="utf-8") as file:
    json.dump(run_manifest, file, indent=2, ensure_ascii=False)

written = [path for path in EMITTED_PATHS if path.exists()]
missing = [str(path) for path in EMITTED_PATHS if not path.exists()]
if missing:
    raise RuntimeError(f"Expected outputs were not written: {missing}")
for path in written:
    if OUT_DIR not in path.parents:
        raise RuntimeError(f"An output escaped the disposable directory: {path}")

print("Appendix variant-comparison study completed.")
print("Cases:", len(base_df), "| variants:", len(VARIANT_KEYS), "| methods:", len(METHOD_KEYS))
print("Sealed artifact writes:", run_manifest["sealed_artifact_writes"])
print("Output directory:", OUT_DIR)
for path in written:
    print("  wrote:", path.name)


Appendix variant-comparison study completed.
Cases: 2288 | variants: 4 | methods: 4
Sealed artifact writes: 0
Output directory: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/_disposable_query_variant_comparison
  wrote: variant_generated_queries_face.parquet
  wrote: variant_text_audit_face.csv
  wrote: variant_leakage_flag_audit_face.csv
  wrote: variant_per_query_metrics_face.parquet
  wrote: variant_method_pool_depth_summary.csv
  wrote: variant_method_headline_hitrate.csv
  wrote: variant_runtime_components.csv
  wrote: variant_method_runtime.csv
  wrote: variant_paired_inference_vs_C.csv
  wrote: variant_C_reproduction_report.csv
  wrote: variant_comparison_run_manifest.json
